# Projet *Speed Dating avec Tinder* – Certification CDSD – bloc 2

<img src="https://full-stack-assets.s3.eu-west-3.amazonaws.com/M03-EDA/Tinder-Symbole.png" alt="Tinder" width="384" height="216">

Auteur : Yoann ROBERT

Date de la présentation : 16 juin 2026

## Introduction

### Contexte

L'équipe Marketing de Tinder a noté une baisse dans le nombre de matches. Elle souhaite en comprendre les raisons et trouver ce qui intéresse une personne chez une autre.

Pour tenter de répondre à ce questionnement, elle a mené une campagne d'expériences de speed dating. Des candidats de sexes opposés se rencontrent et remplissent un questionnaire. Chaque ligne du [dataset](https://full-stack-assets.s3.eu-west-3.amazonaws.com/M03-EDA/Speed+Dating+Data.csv) fourni correspond à une rencontre de quatre minutes et contient notamment la décision d'aller à un second *date* donnée secrètement par un candidat pour son partenaire.

### Dataset

Une description complète du dataset est fournie [ici](https://full-stack-assets.s3.eu-west-3.amazonaws.com/M03-EDA/Speed+Dating+Data+Key.doc).

Nous y apprenons que le questionnaire est rempli par les candidats en plusieurs temps : à l'inscription, en début de soirée, en milieu de soirée, le lendemain et 3-4 semaines après. La correspondance entre le suffixe des variables et sa signification est la suivante :

| Suffixe | Instant                                      |
|---------|----------------------------------------------|
| _1      | Au moment de l'inscription                   |
| _s      | À la moitié de la soirée de speed dating     |
| _2      | Au lendemain de la soirée de speed dating    |
| _3      | 3-4 semaines après la soirée de speed dating |

À ces différents moments, le candidat note six attributs :
- attirance physique
- sincérité
- intelligence
- fun\*
- ambition
- intérêts partagés.

\* *Nous pourrions traduire **fun** par humour, amusement, côté divertissant, côté sympathique, ou d'autres mots, mais nous le laisserons tel quel car il n'a pas une traduction littérale exacte en français.*

Pour ceux-ci, le candidat répond :
- ce qu'il recherche chez un partenaire (du sexe opposé),
- ce qu'il pense qu'un candidat du sexe opposé recherche chez un partenaire,
- comment il s'auto-évalue sur ces critères,
- ce qu'il pense que les candidats du même sexe recherchent chez un partenaire du sexe opposé,
- comment les candidats du sexe opposé l'évaluent
- et quelle est l'importance de chaque attribut selon lui sur la décision d'aller à un second date.

Les autres champs du dataset contiennent, en partie, les éléments suivants :
- les identifiants relatifs à l'événement (candidat, partenaire, **session** de dating, etc.)
- les données démographiques ou sociologiques (âge, sexe, origine ethnique\*\*, religion, lieu d'origine, revenus, etc.),
- les habitudes concernant le dating,
- les centres d'intérêt.

\*\* *Le mot employé dans le dataset et dans la documentation est "race" mais nous l'excluons de notre vocabulaire.
Nous pourrions utiliser le mot "origine ethnique" pour employer une expression plus convenable,
mais nous préférons simplifier par l'utilisation du mot "origine", moins connoté et plus politiquement correct.*

Pour alléger l'écriture, mais surtout la lecture, nous utilisons le masculin générique ("le candidat", "son partenaire") et pas une écriture avec des expressions comme "le/la candidat(e)", "son/sa partenaire", etc., qui fatiguerait l'œil du lecteur.

### Objectifs

Utiliser le dataset pour comprendre ce qui attire un candidat chez un partenaire pour qu'il accepte d'aller à un second date, au moyen de :
- statistiques descriptives
- visualisations
- légendes et interprétations

### Indications

Pour nous aider, Jedha propose d'explorer les questions suivantes :

- Quels sont les attributs les moins désirables chez un homme ? Est-ce différent chez une femme ?
- À quel point les gens pensent que l'attirance physique est importante dans la sélection d'un partenaire vs. son impact réel ?
- Est-ce que les intérêts partagés sont plus importants que d'être de la même origine ?
- Est-ce que les gens peuvent prédire précisément leur propre valeur dans le marché du dating ?
- Pour obtenir un second date, est-ce qu'il est préférable d'être le premier speed date de quelqu'un ou le dernier ?

### Livrable

Ce notebook.

### Plan de l'étude

L'étude se déroule selon le plan suivant :
- Évaluation de la qualité des données
- Partie 1 — Analyse des participants présents dans l'étude
- Partie 2 — Facteurs déclarés pour l'attirance
- Partie 3 — Facteurs réels pour l'attirance
- Partie 4 — Facteurs contextuels pour l'attirance
- Partie 5 — Évaluation des attributs : auto-évaluation vs évaluations reçues par les partenaires
- Conclusion et recommandations pour Tinder

## Configuration

### Imports des libraries

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from itertools import product
from plotly.subplots import make_subplots
from statsmodels.nonparametric.smoothers_lowess import lowess

### Fonctions

In [2]:
def find_column_names(
        df_: pd.DataFrame | pd.Series,
        startswith: list[str] | None = None,
        endswith: list[str] | None = None
) -> list[str]:
    """
    Return column names (or index labels) that match all combinations of
    given prefixes and suffixes.

    If neither `startswith` nor `endswith` is provided, all column names
    are returned. Otherwise, only names formed by concatenating a prefix with
    a suffix (i.e. `prefix + suffix`) that actually exist in the DataFrame
    or Series are returned.

    Parameters
    ----------
    df_ : pd.DataFrame or pd.Series
        The object to search. For a DataFrame, column names are used;
        for a Series, index labels are used.
    startswith : list of str, optional
        List of prefixes. If None, defaults to `[""]` (no prefix filter).
    endswith : list of str, optional
        List of suffixes. If None, defaults to `[""]` (no suffix filter).

    Returns
    -------
    list of str
        Column/index names matching at least one (prefix, suffix) pair,
        in the order produced by their Cartesian product.
    """

    sw: list[str] = list(startswith) if startswith is not None else [""]
    ew: list[str] = list(endswith) if endswith is not None else [""]

    if "" in sw and "" in ew:
        return df_.columns.tolist()

    columns = df_.columns if isinstance(df_, pd.DataFrame) else df_.index
    columns = [s + e for s, e in product(sw, ew) if s + e in columns]

    return columns


def get_column_min_max(df_: pd.DataFrame, column_name: str) -> tuple[float, float]:
    """
    Return the minimum and maximum values of a DataFrame column.

    Parameters
    ----------
    df_ : pd.DataFrame
        The DataFrame containing the column to inspect.
    column_name : str
        Name of the column to compute the min and max from.

    Returns
    -------
    tuple of float
        A (min, max) tuple of the column values.
    """

    col_no_NaN = df_[column_name].dropna(how="all").copy()
    col_values = np.asarray(col_no_NaN, dtype=float)
    return col_values.min(), col_values.max()


def clip_values(
        df_: pd.DataFrame,
        startswith: list[str] | None = None,
        endswith: list[str] | None = None,
        min_value: int=1,
        max_value: int=10,
        verbose: bool=False
) -> pd.DataFrame:
    """
    Clip values of selected columns to a given range.

    Columns to clip are selected by matching `startswith` and `endswith`
    patterns via `find_column_names`. The original DataFrame is not modified.

    Parameters
    ----------
    df_ : pd.DataFrame
        The DataFrame to process.
    startswith : list of str, optional
        List of prefixes used to select columns. If None, no prefix filter
        is applied.
    endswith : list of str, optional
        List of suffixes used to select columns. If None, no suffix filter
        is applied.
    min_value : int, optional
        Lower bound of the clipping range. Default is 1.
    max_value : int, optional
        Upper bound of the clipping range. Default is 10.
    verbose : bool, optional
        If True, prints min and max values before and after clipping for
        each column. Default is False.

    Returns
    -------
    pd.DataFrame
        A copy of `df_` with selected columns clipped to [`min_value`, `max_value`].
    """

    df_ = df_.copy()
    columns_to_clip = find_column_names(df_, startswith, endswith)

    for c in columns_to_clip:
        if verbose:
            col_min, col_max = get_column_min_max(df_, c)
            print(f"\nClipping column '{c}' between {min_value} and {max_value}...")
            print(f"\tValues before clipping\t\tmin: {col_min}\t\tmax: {col_max}")
        df_[c] = df_[c].clip(min_value, max_value)
        if verbose:
            col_min, col_max = get_column_min_max(df_, c)
            print(f"\tValues after clipping\t\tmin: {col_min}\t\tmax: {col_max}")
            print(f"Done.")

    return df_


def round_values(
        df_in: pd.DataFrame,
        startswith: list[str] | None = None,
        endswith: list[str] | None = None,
        verbose: bool=False
) -> pd.DataFrame:
    """
    Round values of selected columns to the nearest integer.

    Columns to round are selected by matching `startswith` and `endswith`
    patterns via `find_column_names`. The original DataFrame is not modified.

    Parameters
    ----------
    df_in : pd.DataFrame
        The DataFrame to process.
    startswith : list of str, optional
        List of prefixes used to select columns. If None, no prefix filter
        is applied.
    endswith : list of str, optional
        List of suffixes used to select columns. If None, no suffix filter
        is applied.
    verbose : bool, optional
        If True, prints the name of each column being rounded. Default is False.

    Returns
    -------
    df_out: pd.DataFrame
        A copy of `df_` with selected columns rounded to the nearest integer.
    """

    df_out = df_in.copy()
    columns_to_round = find_column_names(df_out, startswith, endswith)
    for c in columns_to_round:
        if verbose:
            print(f"\nRounding values of column '{c}' to the nearest integer...")
        df_out[c] = df_out[c].round()
    return df_out


def wrap_label(label, max_chars=10):
    words = label.split()
    lines, current = [], ""
    for word in words:
        if len(current) + len(word) > max_chars and current:
            lines.append(current.strip())
            current = word + " "
        else:
            current += word + " "
    lines.append(current.strip())
    return "<br>".join(lines)


# To convert ratings from 1→10 to 0→100, we need to check for NaNs across the related 6 attributes.
#
# For the ratings that need to be converted from 1→10 to 0→100,
# the possibilities and the associated actions are:
#     1) All NaNs or no NaNs → nothing to do
#     2) At least one NaN (but not all NaNs) → NaNs arbitrarily converted to 1 (the lowest rating)
#
# For the ratings that are already given as 0→100,
# we will do as follows.
#     1) All NaNs → nothing to do
#     2) 0 <= nb NaNs < 6 → NaNs replaced by 0, then check if the sum of the attributs equals to 100
#         a) yes → nothing left to do
#         b) no → rescaling of the attributes to have their sum equal to 100
def preprocess_ratings(df_in: pd.DataFrame, endswith: str, attribute_abbreviations: list[str]) -> pd.DataFrame:
    columns = [attr_ + endswith for attr_ in attribute_abbreviations if attr_ + endswith in df_in.columns]
    df_out = df_in.copy()
    subset_ = df_in[columns].copy()

    all_nan = subset_.isna().all(axis=1)
    max_vals = subset_.max(axis=1)

    # Scale 1-10: fill NaN with 1
    low = (~all_nan) & (max_vals <= 10)
    subset_.loc[low] = subset_.loc[low].fillna(1)

    # Scale 0-100: fill NaN with 0, then renormalize rows whose sum ≈ 100
    high = (~all_nan) & (max_vals > 10)
    subset_.loc[high] = subset_.loc[high].fillna(0)
    row_sums = subset_.loc[high].sum(axis=1)
    renorm_idx = row_sums.index[(row_sums - 100).abs() < 1]
    if len(renorm_idx):
        s = subset_.loc[renorm_idx].sum(axis=1)
        subset_.loc[renorm_idx] = subset_.loc[renorm_idx].div(s, axis=0) * 100

    df_out[columns] = subset_
    return df_out


# Converting ratings from 1→10 to 0→100 is not perfect.
# Furthermore, in one case or another, the participant is not subject to the same cognitif process.
# Some properties should be ensured:
#     - A rating of 1/10 should receive 0 points.
#     - A rating of 10/10 does not ensure 100 points. The number of points for an attribute depends on
#     the ratings of the other attributes (to have 100 points, the remaining 5 must all have a 1/10 rating).
#     - A higher rating has always more points than a lower one (monotony principle).
#     - A rating twice as high as another one receives exactly double the points (proportionality principle).
#     - Degenerate case: if all notes are 1/10, they do not receive 0 points but 100/6=16,7 points.
def normalize_ratings(df_in: pd.DataFrame, endswith: str, attribute_abbreviations: list[str]) -> pd.DataFrame:
    df_out = preprocess_ratings(df_in=df_in, endswith=endswith, attribute_abbreviations=attribute_abbreviations)
    columns = [attr_ + endswith for attr_ in attribute_abbreviations if attr_ + endswith in df_in.columns]
    subset_ = df_out[columns].copy()
    n = len(columns)

    all_nan = subset_.isna().all(axis=1)
    max_vals = subset_.max(axis=1)

    # All values == 1: uniform distribution
    all_ones = (~all_nan) & (subset_ == 1).all(axis=1)
    subset_.loc[all_ones] = 100 / n

    # Scale 1-10: (x - 1) / (sum - n) * 100
    low = (~all_nan) & (~all_ones) & (max_vals <= 10)
    if low.any():
        s = subset_.loc[low].sum(axis=1)
        subset_.loc[low] = subset_.loc[low].sub(1).div(s - n, axis=0) * 100

    # Scale 0-100: x / sum * 100
    high = (~all_nan) & (~all_ones) & (max_vals > 10)
    if high.any():
        s = subset_.loc[high].sum(axis=1)
        subset_.loc[high] = subset_.loc[high].div(s, axis=0) * 100

    df_out[columns] = subset_
    return df_out

### Constantes

In [3]:
# Constants
ATTR_ABBREVS = ["attr", "sinc", "intel", "fun", "amb", "shar"]
ATTR_LABELS = {
    'attr': 'Attirance physique',
    'sinc': 'Sincérité',
    'intel': 'Intelligence',
    'fun': 'Fun',
    'amb': 'Ambition',
    'shar': 'Intérêts partagés'
}

### Personnalisation

In [4]:
GENDER_LABELS = {0: "Femmes", 1: "Hommes"}
GENDER_COLORS = {0: "hotpink", 1: "royalblue"}
GENDER_COLORS_FROM_LABELS = {v: GENDER_COLORS[i] for i, v in GENDER_LABELS.items()}
RATE_COLORS = {"dec": "coral", "match": "red"}
DEFAULT_PLOT_COLOR = "tomato"
ATTENTION_PLOT_COLOR = "orchid"
ALTERNATIVE_PLOT_COLOR = "sandybrown"
CONTINUOUS_COLOR_SCALE = px.colors.sequential.Reds

# Modèle personnalisé pour Plotly
tinder_template = go.layout.Template(
    layout=go.Layout(
        margin=dict(t=50, b=50, l=50, r=50),
        width=1000,
        font=dict(size=10),
        annotationdefaults=dict(font=dict(size=16)),
        title=dict(font=dict(size=18)),
        xaxis=dict(title=dict(font=dict(size=14)), tickfont=dict(size=12)),
        yaxis=dict(title=dict(font=dict(size=14)), tickfont=dict(size=12)),
        legend=dict(font=dict(size=12))
    )
)
pio.templates["tinder_template"] = tinder_template
pio.templates.default = "plotly_dark+tinder_template"

---

## Évaluation de la qualité des données

Cette partie a pour buts :
- aperçu du dataset,
- analyse des valeurs manquantes,
- analyse des valeurs aberrantes,
- pré-traitement avant poursuite de l'étude.

### Aperçu du dataset

Après téléchargement du fichier CSV depuis un bucket AWS S3 et import directement dans un DataFrame, nous relevons que le dataset possède :
- 8378 lignes
- 195 colonnes.

In [5]:
CSV_PATH = 'https://full-stack-assets.s3.eu-west-3.amazonaws.com/M03-EDA/Speed+Dating+Data.csv'
df = pd.read_csv(CSV_PATH, encoding='ISO-8859-1')

nb_rows, nb_cols = df.shape
print(f"Nombre de lignes :\t\t{nb_rows}\t(i.e., number of speed dates)\n" +
      f"Nombre de colonnes :\t{nb_cols}\n")

print("Aperçu du dataset (5 premières lignes) :")
df.head(5)

Nombre de lignes :		8378	(i.e., number of speed dates)
Nombre de colonnes :	195

Aperçu du dataset (5 premières lignes) :


,iid,id,gender,idg,condtn,wave,round,position,positin1,order,...,attr3_3,sinc3_3,intel3_3,fun3_3,amb3_3,attr5_3,sinc5_3,intel5_3,fun5_3,amb5_3
0,1,1.0,0,1,1,1,10,7,NaN,4,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
1,1,1.0,0,1,1,1,10,7,NaN,3,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
2,1,1.0,0,1,1,1,10,7,NaN,10,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
3,1,1.0,0,1,1,1,10,7,NaN,5,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
4,1,1.0,0,1,1,1,10,7,NaN,7,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN


### Valeurs manquantes

La représentation graphique du dataset ci-dessous indique visuellement où sont situées les valeurs manquantes.

In [6]:
fig = go.Figure(
    go.Heatmap(
        z=df.isna().astype(int).values,
        x=df.columns.tolist(),
        y=df.index.tolist(),
        colorscale=[[0, 'black'], [1, 'white']],
        showscale=False
    )
)

fig.update_layout(
    height=800,
    title=dict(
        text="Représentation des données manquantes (présentes en noir, manquantes en blanc)",
    )
)
fig.update_xaxes(
    title="Nom de colonne"
)
fig.update_yaxes(
    autorange='reversed',
    title="Indice de ligne"
)
fig.show()

Nous pouvons noter plusieurs caractéristiques sur les valeurs manquantes du dataset :
- Plus nous allons vers la droite, plus il manque des données. Cela correspond **au manque progressif de réponses de la part des candidats** : plus proche de l'inscription une question se trouve, moins les données sont manquantes, et vice versa.
- Nous remarquons des bandes blanches verticales indiquant potentiellement que **des questions n'ont pas toujours été posées** :
    - Si une bande démarre tout en haut puis se termine avant d'arriver tout en bas, alors cela veut dire que la/les questions (selon l'épaisseur de la bande) qui s'y réfèrent n'étaient pas présentes au tout début de la campagne expérimentale (exemple : colonnes `*4_1` ou `shar2_3` comme démontré ci-après.)
    - À l'inverse si elle démarre tout du bas et se termine avant d'arriver tout en haut, alors la/les questions ont disparu du formulaire en cours de campagne (exemple : colonne `expnum`)
    - Si une tendance (même imparfaite) d'alternance entre les deux cas précédents, alors cela veut dire que les questions ont été absentes du formulaire pour certaines sessions (exemple : les questions posées en milieu de soirée).

In [7]:
# Illustration : vérification de l'introduction de la question "shar2_3" dans le formulaire

waves_and_NaNs_for_shar2_3 = df[["wave", "shar2_3"]].value_counts(dropna=False).reset_index().drop(columns="count")
waves_without_all_NaNs_for_shar2_3 = (
    waves_and_NaNs_for_shar2_3[~waves_and_NaNs_for_shar2_3["shar2_3"].isna()]
    ["wave"]
    .drop_duplicates()
    .sort_values()
    .astype(int)
    .tolist()
)
print(f"Sessions qui ont des valeurs autres que NaN pour 'shar2_3' : {waves_without_all_NaNs_for_shar2_3}")
print(f"La question associée à la colonne 'shar2_3' a été introduite pour la session #{waves_without_all_NaNs_for_shar2_3[0]}.")

Sessions qui ont des valeurs autres que NaN pour 'shar2_3' : [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
La question associée à la colonne 'shar2_3' a été introduite pour la session #10.


#### Vision par ligne

In [8]:
stats_row_NaNs = pd.DataFrame(
    df.isna().sum(axis=1).sort_values(ascending=False),
    columns=['nb_values']
)
stats_row_NaNs["Taux de données manquantes par ligne (%)"] = (stats_row_NaNs["nb_values"] / nb_cols * 100).round(1)
stats_row_NaNs["Taux de données présentes par ligne (%)"] = 100 - stats_row_NaNs["Taux de données manquantes par ligne (%)"]

summarised_stats_complete_rows = pd.DataFrame(
    [completeness_rate for completeness_rate in np.linspace(0, 100, 1000)],
    columns=['Taux de données présentes par ligne (%)']
)

summarised_stats_complete_rows["Nombre de lignes (% du total)"] = (
    summarised_stats_complete_rows['Taux de données présentes par ligne (%)'].apply(
        lambda x: round(
            stats_row_NaNs[stats_row_NaNs["Taux de données présentes par ligne (%)"] <= x].shape[0] / nb_rows * 100,
            1
        )
    )
)

fig = px.line(
    summarised_stats_complete_rows,
    y = "Taux de données présentes par ligne (%)",
    x = 'Nombre de lignes (% du total)',
    title = "Taux de données présentes par ligne vs nombre de lignes",
    color_discrete_sequence=[DEFAULT_PLOT_COLOR],
)
fig.update_traces(marker=dict(size=10))
fig.update_layout(
    yaxis_scaleanchor="x",
    yaxis_scaleratio=1,
    width=600, height=600
)
fig.show()

#### Vision par colonne

In [9]:
missing_data = df.isna().sum().sort_values(ascending=False)
missing_data = pd.DataFrame(missing_data, columns=['nb_values'])
missing_data["pct"] = (missing_data["nb_values"] / nb_rows * 100).round(1)

In [10]:
stats_column_NaNs_above_50pct = missing_data[missing_data['pct'] >= 50].sort_values(by='pct', ascending=False)

print("Nombre de colonnes avec des données manquantes > 50% :", end="\t\t")
print(f"{len(stats_column_NaNs_above_50pct)} / {nb_cols}", end="\t")
print(f"({round(len(stats_column_NaNs_above_50pct) / nb_cols * 100, 1)} %)\n")

print(f"Top 10 des colonnes avec le plus de données manquantes par ligne :")
stats_column_NaNs_above_50pct["pct"].head(10)

Nombre de colonnes avec des données manquantes > 50% :		59 / 195	(30.3 %)

Top 10 des colonnes avec le plus de données manquantes par ligne :


num_in_3    92.0
numdat_3    82.1
expnum      78.5
sinc7_2     76.7
amb7_2      76.7
shar7_2     76.4
attr7_2     76.3
intel7_2    76.3
fun7_2      76.3
shar7_3     75.9
Name: pct, dtype: float64

In [11]:
fig = px.bar(
    stats_column_NaNs_above_50pct,
    x='pct',
    color='pct',
    title="Colonnes avec un taux de données manquantes > 50% (% du nombre de lignes)",
    width=1000,
    height=1200,
    range_x=[0, 100],
    labels={
        'index': 'Nom de colonne',
        'pct': 'Taux de données manquantes (%)'
    },
    color_continuous_scale=CONTINUOUS_COLOR_SCALE
)
#fig.update_layout(font=dict(size=10))
fig.show()

#### Résumé

- Nous avons pris une limite relativement permissive en nombre de valeurs manquantes par colonne de 50%, comme critère de validité de la présence des colonnes dans le dataset.
- Dans le dataset, 59 colonnes (soit 30% des colonnes) dépassent cette limite.
- Les colonnes avec le plus de valeurs manquantes sont celles remplies par les candidats au lendemain de la soirée et plusieurs semaines après, ainsi que deux colonnes remplies à l'inscription avec des importances assez faibles pour notre étude : le montant des frais de scolarité et le score du SAT (examen standardisé aux États-Unis utilisé pour l'admission dans les universités).
- Ces valeurs manquantes ne devraient pas être gênantes pour conduire notre étude dans sa majeure partie. En revanche, il faudra en tenir compte dans la partie 5.
- Par ailleurs, sur la représentation graphique des valeurs manquantes dans le dataset et sur la sortie des pourcentages de lignes incomplètes, nous pouvons constater que
    - la quasi-totalité des lignes (>90%) sont incomplètes (i.e., ont au moins 5% de valeurs manquantes),
    - au moins la moitié des lignes (>55%) présentent au moins 25% de valeurs manquantes,
    - mais que les valeurs manquantes sur ces lignes sont concentrées sur les colonnes citées précédemment.
- Pour ces raisons, nous faisons le choix de ne supprimer aucune ligne ni colonne du dataset de manière globale. Les valeurs manquantes seront retirées au cas par cas, selon les analyses particulières que nous ferons. Elles pourront aussi être ponctuellement remplacées par des valeurs nulles ou adéquates.

### Valeurs aberrantes

Nous avons remarqué la présence de certaines valeurs aberrantes. Elles concernent en particulier les notes censées être entre 1 et 10 (inclus). Pour pallier ce problème, nous travaillons avec deux sous-ensembles disjoints du dataset : un pour les sessions de 6 à 9, un autre pour les autres sessions.

In [12]:
mask_waves6to9 = df["wave"].isin([6, 7, 8, 9])
df_waves6to9 = df[mask_waves6to9]
df_wavesNot6to9 = df[~mask_waves6to9]

#### Sessions 6-9

In [13]:
print("Statistiques pour les colonnes numériques (pour les sessions 6 à 9) :")
df_waves6to9.describe()

Statistiques pour les colonnes numériques (pour les sessions 6 à 9) :


,iid,id,gender,idg,condtn,wave,round,position,positin1,order,...,attr3_3,sinc3_3,intel3_3,fun3_3,amb3_3,attr5_3,sinc5_3,intel5_3,fun5_3,amb5_3
count,1562.000000,1562.000000,1562.00000,1562.000000,1562.000000,1562.000000,1562.000000,1562.000000,1562.000000,1562.000000,...,943.000000,943.000000,943.000000,943.000000,943.000000,0.0,0.0,0.0,0.0,0.0
mean,188.838028,8.996159,0.50000,17.428297,1.839949,8.120359,16.928297,8.964149,8.964149,8.964149,...,8.076352,8.528102,9.304348,8.227996,8.221633,NaN,NaN,NaN,NaN,NaN
std,28.739971,5.376175,0.50016,10.798203,0.366771,0.976788,3.978157,5.393307,5.393307,5.393307,...,1.737712,1.866285,1.659961,2.052387,2.114724,NaN,NaN,NaN,NaN,NaN
min,132.000000,1.000000,0.00000,1.000000,1.000000,6.000000,5.000000,1.000000,1.000000,1.000000,...,4.000000,6.000000,6.000000,4.000000,3.000000,NaN,NaN,NaN,NaN,NaN
25%,163.000000,4.000000,0.00000,8.000000,2.000000,7.000000,16.000000,4.000000,4.000000,4.000000,...,7.000000,7.000000,8.000000,7.000000,7.000000,NaN,NaN,NaN,NaN,NaN
50%,194.000000,8.500000,0.50000,16.000000,2.000000,9.000000,20.000000,8.000000,8.000000,8.000000,...,8.000000,8.000000,9.000000,8.000000,8.000000,NaN,NaN,NaN,NaN,NaN
75%,214.000000,13.000000,1.00000,26.000000,2.000000,9.000000,20.000000,13.000000,13.000000,13.000000,...,9.000000,9.000000,9.000000,9.000000,9.000000,NaN,NaN,NaN,NaN,NaN
max,233.000000,20.000000,1.00000,40.000000,2.000000,9.000000,20.000000,20.000000,20.000000,20.000000,...,12.000000,12.000000,12.000000,12.000000,12.000000,NaN,NaN,NaN,NaN,NaN


Ici `X` désigne n'importe quelle abréviation parmi `attr`, `sinc`, `intel`, `fun`, `amb` et `shar`.

Remarques :
- Des notes qui devraient être données entre 1 et 10, comme indiqué dans la documentation, sont parfois de 0 ou supérieures à 10. Les valeurs inférieures à 1 et celles supérieures à 10 seront, respectivement, substituées par 1 et 10. Cela concerne `X`, `X_o`, `X3_3` et les 17 notes concernant les activités.
- Certaines notes devraient être données entre 1 et 10 et, par conséquent, devraient être converties entre 0 et 100 (distribution de 100 points entre les six attributs) pour rester cohérent avec les autres sessions.
    - Cependant, certaines de ces notes ont déjà été converties. Cela concerne `X1_1`, `X2_1`, `X1_2` et `X1_3`.
    - Les colonnes qui nécessiteraient d'être converties entre 0 et 100 sont `X4_1`, `X4_2`, `X4_3` et `X2_3`.
        - Pour les trois premiers groupes, les questions concernent ce que le candidat pense que les candidats du même sexe recherchent chez un partenaire du sexe opposé. Notre étude intègre cette question (en réalité seulement `X4_1`) donc nous convertirons les notes de 1→10 en 0→100.
        - Le dernier groupe de colonnes, `X2_3`, correspond aux questions posées plusieurs semaines après le speed dating et, plus spécifiquement, concerne ce que le candidat pense que le sexe opposé recherche chez un partenaire. Cette question, posée à ce moment précis de la campagne, et par ailleurs n'ayant reçu que très peu de réponses par les candidats, sera écartée de notre étude et les colonnes associées resteront inchangées.
    - Des colonnes, données entre 0 et 100 alors qu'elles doivent être entre 1 et 10, resteront inchangées par cohérence avec les autres sessions. Cela concerne `X1_s`.
- Les colonnes `shar3_1`, `shar3_s`, `shar3_2`, `shar3_3`, `shar5_1`, `shar5_2` et `shar5_3` n'existent pas, car il n'y aurait pas de sens à demander à un candidat à quel point il estime qu'il a des intérêts partagés en général (et non avec quelqu'un en particulier) ou, à l'inverse, à quel point il estime que les candidats du sexe opposé (sans les désigner individuellement les uns les autres) l'évaluent sur les intérêts partagés.
- La colonne `shar2_3` ne contient que des valeurs manquantes (idem). Comme montré ci-dessus, c'est normal. En effet, la question qui s'y réfère n'a été ajoutée qu'à partir de la dixième session.
- Des notes (données entre 1 et 10) contiennent des demi-points. Bien qu'il n'ait pas été spécifié dans la documentation qu'elles doivent être des valeurs entières, nous arrondirons ces valeurs à l'entier le plus proche, par souci de simplification de l'analyse. Nous n'appliquerons pas d'arrondi sur les notes en 0 et 100, car cela est inutile.

Résumé des actions à mener :
- Rogner les notes normalement données entre 1 et 10 puis les arrondir à l'entier le plus proche. Cela concerne : `X`, `X_o`, `X3_1`, `X4_1`, `X5_1`, `X3_s`, `X3_2`, `X4_2`, `X5_2`, `X2_3`, `X3_3`, `X4_3`, `X5_3`, `like`, `like_o`, `prob`, `prob_o`, `imprace`, `imprelig`, `exphappy` et les 17 activités. Cette liste est surchargée par sécurité (appliquer ces actions sur certaines colonnes ne produira aucune modification).
- Ensuite, convertir les notes 1→10 en 0→100. Cela concerne : `X4_1`, `X4_2` et `X4_3`.

In [14]:
activity_columns = df.columns.tolist()[50: 50 + 16 + 1]
print(f"{len(activity_columns)} activités: {", ".join(activity_columns)}")

17 activités: sports, tvsports, exercise, dining, museums, art, hiking, gaming, clubbing, reading, tv, theater, movies, concerts, music, shopping, yoga


In [15]:
starts_with = ATTR_ABBREVS + ["like", "prob", "imprace", "imprelig", "exphappy"] + activity_columns
ends_with = ["", "_o", "3_1", "4_1", "5_1", "3_s", "3_2", "4_2", "5_2", "2_3", "3_3", "4_3", "5_3"]
df_waves6to9 = clip_values(df_waves6to9, startswith=starts_with, endswith=ends_with)
df_waves6to9 = round_values(df_waves6to9, startswith=starts_with, endswith=ends_with)

print("Statistiques pour les colonnes numériques (pour les vagues 6 à 9) – après écrêtage puis arrondi des données :")
df_waves6to9.describe()

Statistiques pour les colonnes numériques (pour les vagues 6 à 9) – après écrêtage puis arrondi des données :


,iid,id,gender,idg,condtn,wave,round,position,positin1,order,...,attr3_3,sinc3_3,intel3_3,fun3_3,amb3_3,attr5_3,sinc5_3,intel5_3,fun5_3,amb5_3
count,1562.000000,1562.000000,1562.00000,1562.000000,1562.000000,1562.000000,1562.000000,1562.000000,1562.000000,1562.000000,...,943.000000,943.000000,943.000000,943.000000,943.000000,0.0,0.0,0.0,0.0,0.0
mean,188.838028,8.996159,0.50000,17.428297,1.839949,8.120359,16.928297,8.964149,8.964149,8.964149,...,7.838812,8.161188,8.810180,7.903499,7.909862,NaN,NaN,NaN,NaN,NaN
std,28.739971,5.376175,0.50016,10.798203,0.366771,0.976788,3.978157,5.393307,5.393307,5.393307,...,1.254080,1.237892,0.912884,1.518540,1.624957,NaN,NaN,NaN,NaN,NaN
min,132.000000,1.000000,0.00000,1.000000,1.000000,6.000000,5.000000,1.000000,1.000000,1.000000,...,4.000000,6.000000,6.000000,4.000000,3.000000,NaN,NaN,NaN,NaN,NaN
25%,163.000000,4.000000,0.00000,8.000000,2.000000,7.000000,16.000000,4.000000,4.000000,4.000000,...,7.000000,7.000000,8.000000,7.000000,7.000000,NaN,NaN,NaN,NaN,NaN
50%,194.000000,8.500000,0.50000,16.000000,2.000000,9.000000,20.000000,8.000000,8.000000,8.000000,...,8.000000,8.000000,9.000000,8.000000,8.000000,NaN,NaN,NaN,NaN,NaN
75%,214.000000,13.000000,1.00000,26.000000,2.000000,9.000000,20.000000,13.000000,13.000000,13.000000,...,9.000000,9.000000,9.000000,9.000000,9.000000,NaN,NaN,NaN,NaN,NaN
max,233.000000,20.000000,1.00000,40.000000,2.000000,9.000000,20.000000,20.000000,20.000000,20.000000,...,10.000000,10.000000,10.000000,10.000000,10.000000,NaN,NaN,NaN,NaN,NaN


Il a été remarqué que des colonnes comme `met`, `met_o` et `date_3`, devant posséder des valeurs binaires (`1` pour "Yes", `2` pour "No"), ont des valeurs aberrantes :
- selon les sessions, les valeurs de `met` et `met_o` peuvent être dans `{1, 2}` mais aussi dans `{0, 1}` et même prendre des valeurs comme `7`et `8`.
- les valeurs de `date_3` sont dans `{0, 1}`.

Nous ne ferons pas de tentative d'ajustement pour `met`, `met_o` et ces colonnes seront simplement omises de l'étude.
Nous ferons une hypothèse sur la traduction des valeurs prises par `date_3` en partie 4.

In [16]:
print("Manque de consistance des valeurs pour les colonnes `met` et `met_o`:\n")
print(df_waves6to9.groupby("wave")["met"].value_counts().sort_index().to_string())

Manque de consistance des valeurs pour les colonnes `met` et `met_o`:

wave  met
6     1.0     10
      2.0     39
7     0.0    464
      1.0     45
8     1.0     13
      2.0    179
      6.0      1
      7.0      1
      8.0      1
9     1.0     46
      2.0    720
      7.0      1


#### Sessions 1-5 et 10-21

In [17]:
print("Statistiques pour les colonnes numériques (sessions 1-5 et 10-21) :")
df_wavesNot6to9.describe()

Statistiques pour les colonnes numériques (sessions 1-5 et 10-21) :


,iid,id,gender,idg,condtn,wave,round,position,positin1,order,...,attr3_3,sinc3_3,intel3_3,fun3_3,amb3_3,attr5_3,sinc5_3,intel5_3,fun5_3,amb5_3
count,6816.000000,6815.000000,6816.000000,6816.000000,6816.000000,6816.000000,6816.000000,6816.000000,4970.000000,6816.000000,...,3031.000000,3031.000000,3031.000000,3031.000000,3031.000000,2016.000000,2016.000000,2016.000000,2016.000000,2016.000000
mean,305.409624,8.952018,0.500734,17.303991,1.826291,12.091256,16.859155,9.060739,9.400000,8.919308,...,6.980205,7.958100,8.103926,7.481689,7.133289,6.810020,7.615079,7.932540,7.155258,7.048611
std,167.897804,5.517740,0.500036,10.973791,0.378887,6.405624,4.441200,5.542669,5.725118,5.496369,...,1.426622,1.496716,1.262072,1.596689,1.836569,1.507341,1.504551,1.340868,1.672787,1.717988
min,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,6.000000,1.000000,1.000000,1.000000,...,2.000000,2.000000,3.000000,2.000000,1.000000,2.000000,2.000000,4.000000,1.000000,1.000000
25%,116.000000,4.000000,0.000000,8.000000,2.000000,5.000000,14.000000,4.000000,4.000000,4.000000,...,6.000000,7.000000,8.000000,7.000000,6.000000,6.000000,7.000000,7.000000,6.000000,6.000000
50%,335.000000,8.000000,1.000000,16.000000,2.000000,13.000000,18.000000,8.000000,9.000000,8.000000,...,7.000000,8.000000,8.000000,8.000000,7.000000,7.000000,8.000000,8.000000,7.000000,7.000000
75%,440.250000,13.000000,1.000000,26.000000,2.000000,17.000000,21.000000,13.000000,14.000000,13.000000,...,8.000000,9.000000,9.000000,8.000000,9.000000,8.000000,9.000000,9.000000,8.000000,8.000000
max,552.000000,22.000000,1.000000,44.000000,2.000000,21.000000,22.000000,22.000000,22.000000,22.000000,...,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000


Remarques :
- Des notes qui doivent être données entre 1 et 10 sont parfois nulles ou supérieures à 10. Nous ferons comme indiqué pour les sessions 6 à 9. Cela concerne `X`, `X_o`, les 17 notes concernant les activités et `imprace`.
- Comme évoqué pour les sessions 6 à 9 :
    - des colonnes, données entre 0 et 100 alors qu'elles doivent être entre 1 et 10, resteront inchangées par cohérence avec les autres sessions. Cela concerne `X1_s`.
    - les colonnes `shar3_1`, `shar3_s`, `shar3_2`, `shar3_3`, `shar5_1`, `shar5_2` et `shar5_3` n'existent pas et cela est normal.

Résumé des actions à mener (similaires aux actions pour les sessions 6 à 9) :
- Rogner les notes normalement données entre 1 et 10 puis les arrondir à l'entier le plus proche. Cela concerne : `X`, `X_o`, `X3_1`, `X5_1`, `X3_s`, `X3_2`, `X5_2`, `X3_3`, `X5_3`, `like`, `like_o`, `prob`, `prob_o`, `imprace`, `imprelig`, `exphappy` et les 17 activités. Cette liste est surchargée par sécurité (appliquer ces actions sur certaines colonnes ne produira aucune modification).

In [18]:
starts_with = ATTR_ABBREVS + ["like", "prob", "imprace", "imprelig", "exphappy"] + activity_columns
ends_with = ["", "_o", "3_1", "5_1", "3_s", "3_2", "5_2", "3_3", "5_3"]
df_wavesNot6to9 = clip_values(df_wavesNot6to9, startswith=starts_with, endswith=ends_with)
df_wavesNot6to9 = round_values(df_wavesNot6to9, startswith=starts_with, endswith=ends_with)

print("Statistiques pour les colonnes numériques (sessions 1-5 et 10-21) – après écrêtage puis arrondi des données :")
df_wavesNot6to9.describe()

Statistiques pour les colonnes numériques (sessions 1-5 et 10-21) – après écrêtage puis arrondi des données :


,iid,id,gender,idg,condtn,wave,round,position,positin1,order,...,attr3_3,sinc3_3,intel3_3,fun3_3,amb3_3,attr5_3,sinc5_3,intel5_3,fun5_3,amb5_3
count,6816.000000,6815.000000,6816.000000,6816.000000,6816.000000,6816.000000,6816.000000,6816.000000,4970.000000,6816.000000,...,3031.000000,3031.000000,3031.000000,3031.000000,3031.000000,2016.000000,2016.000000,2016.000000,2016.000000,2016.000000
mean,305.409624,8.952018,0.500734,17.303991,1.826291,12.091256,16.859155,9.060739,9.400000,8.919308,...,6.980205,7.958100,8.103926,7.481689,7.133289,6.810020,7.615079,7.932540,7.155258,7.048611
std,167.897804,5.517740,0.500036,10.973791,0.378887,6.405624,4.441200,5.542669,5.725118,5.496369,...,1.426622,1.496716,1.262072,1.596689,1.836569,1.507341,1.504551,1.340868,1.672787,1.717988
min,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,6.000000,1.000000,1.000000,1.000000,...,2.000000,2.000000,3.000000,2.000000,1.000000,2.000000,2.000000,4.000000,1.000000,1.000000
25%,116.000000,4.000000,0.000000,8.000000,2.000000,5.000000,14.000000,4.000000,4.000000,4.000000,...,6.000000,7.000000,8.000000,7.000000,6.000000,6.000000,7.000000,7.000000,6.000000,6.000000
50%,335.000000,8.000000,1.000000,16.000000,2.000000,13.000000,18.000000,8.000000,9.000000,8.000000,...,7.000000,8.000000,8.000000,8.000000,7.000000,7.000000,8.000000,8.000000,7.000000,7.000000
75%,440.250000,13.000000,1.000000,26.000000,2.000000,17.000000,21.000000,13.000000,14.000000,13.000000,...,8.000000,9.000000,9.000000,8.000000,9.000000,8.000000,9.000000,9.000000,8.000000,8.000000
max,552.000000,22.000000,1.000000,44.000000,2.000000,21.000000,22.000000,22.000000,22.000000,22.000000,...,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000


### Notes à échelles hétérogènes

Nous convertissons les notes de l'échelle 1→10 vers 0→100.

In [19]:
column_suffix_for_normalization = ["1_1", "2_1", "4_1", "1_s", "1_2", "1_3", "4_2", "4_3"]
for ends_with in column_suffix_for_normalization:
    df_waves6to9 = normalize_ratings(df_in=df_waves6to9, endswith=ends_with, attribute_abbreviations=ATTR_ABBREVS)
    df_wavesNot6to9 = normalize_ratings(df_in=df_wavesNot6to9, endswith=ends_with, attribute_abbreviations=ATTR_ABBREVS)

### Concaténation des données après modifications

Après les avoir modifiés, nous concaténons les deux sous-ensembles du dataset.

In [20]:
# PyCharm ne résout pas le type de retour de pd.concat (faux positif), d'où le "ignore" en fin de ligne
df = pd.concat([df_waves6to9, df_wavesNot6to9], axis=0).sort_index()  # type: ignore[union-attr]

### Bilan partiel — Évaluation de la qualité des données

- Dataset volumineux mais hétérogène : 8378 lignes × 195 colonnes.
- Environ 30% des colonnes ont plus de 50% de valeurs manquantes (essentiellement post-soirée).
- Échelles de notation incohérentes entre sessions, harmonisées (1→10 pour chaque attribut vs partage de 100 points entre six attributs).
- Valeurs aberrantes écrêtées, arrondies et converties ; deux colonnes écartées (`met`, `date_3` pour l'analyse binaire).
- Aucune suppression globale de lignes ou colonnes : traitement ciblé au cas par cas.

---

## Partie 1 — Analyse des participants présents dans l'étude

### Genre des participants

Dans ce dataset, nous avons décompté un total de 551 participants distincts : 274 femmes (49,7%) et 277 hommes (50,3%).

In [21]:
participants = df.drop_duplicates('iid').copy()
mask_participants_waves6to9 = participants["wave"].isin([6, 7, 8, 9])

nb_participants = participants.shape[0]
nb_waves = df['wave'].nunique()
nb_males = participants[participants['gender'] == 1].shape[0]
nb_females = participants[participants['gender'] == 0].shape[0]

print(f"Nombre de participants uniques: {nb_participants}")
print(f"Nombre de sessions : {nb_waves}\n")
print(f"Répartition des genres:")
print(f"\tFemmes :  {nb_females}")
print(f"\tHommes :  {nb_males}\n")

participants_pie = (
    participants['gender']
    .map(GENDER_LABELS)
    .value_counts()
    .reset_index()
    .rename(columns={"gender": "Genre", "count": "Effectif"})
)

fig = px.pie(
    participants_pie,
    names="Genre",
    values="Effectif",
    title="Répartition des genres",
    color="Genre",
    color_discrete_map=GENDER_COLORS_FROM_LABELS,
    width=400,
    height=300
)
fig.update_layout(margin=dict(t=50, b=20, l=50, r=20))
fig.show()

Nombre de participants uniques: 551
Nombre de sessions : 21

Répartition des genres:
	Femmes :  274
	Hommes :  277



### Taux de décision positive et taux de match

Le graphique ci-dessous montre que, au cours des différentes soirées de speed dating, le taux moyen de décision de dire "oui" au partenaire pour un second date a été de 42%.
Quant au taux de match, c'est-à-dire un "oui" partagé par les deux candidats, le taux a été de 16,5%, soit en moyenne un match tous les six speed dates.
Il est à noter que les femmes sont plus réticentes à dire "oui" (36,5%) que les hommes (47,4%).

In [22]:
print(
    "Pourcentage de données manquantes pour ces données :\n" +
    (100 * df[['dec', 'match']].isna().sum().div(nb_participants)).round(2).to_string(),
    end="\n\n"
)

yes_and_match_rates = pd.DataFrame(
    df[['dec', 'match']].mean().apply(lambda x: round(100 * x, 1))
).rename({0: "Taux (%)"}, axis=1).rename({"dec": "Taux de 'oui'", "match": "Taux de match"}, axis=0)

yes_rate_per_gender = pd.DataFrame(
    df.groupby('gender')['dec'].mean().apply(lambda x: round(100 * x, 1))
).rename(GENDER_LABELS).rename({"dec": "Taux de 'oui' (%)"}, axis=1)


fig = make_subplots(
    cols=2,
    column_titles=["Taux de 'oui' et de match", "Taux de 'oui' par genre"]
)

fig.add_trace(
    go.Bar(
        x=yes_and_match_rates.index,
        y=yes_and_match_rates["Taux (%)"],
        marker=dict(color=list(RATE_COLORS.values()))
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=yes_rate_per_gender.index,
        y=yes_rate_per_gender["Taux de 'oui' (%)"],
        marker=dict(color=list(GENDER_COLORS_FROM_LABELS.values()))
    ),
    row=1, col=2
)

fig.update_layout(
    showlegend=False,
    height=400, width=600
)

fig.show()


Pourcentage de données manquantes pour ces données :
dec      0.0
match    0.0



### Âge des participants

Dans le dataset, la moyenne d'âge des femmes est de 26,1 ans et celle des hommes est de 26,6 ans.
L'écart entre les deux médianes respectives est plus élevé que celui des moyennes (1 an vers 0,5 an), avec des valeurs de 26,0 et 27,0 ans.
La distribution des âges indique que la plupart des personnes sont des étudiants ou des jeunes adultes.

Parmi la population, huit participants n'ont pas fourni leurs âges et six ont des âges considérés comme des valeurs statistiquement aberrantes.
Nous faisons toutefois le choix de les maintenir dans les analyses.
Dans le premier cas, l'absence de cette donnée peut venir d'une timidité ou d'une discrétion. Exclure ces personnes constituerait un biais de sélection.
Dans le second cas, ces données caractérisent une distribution non centrée et ont alors un intérêt potentiel pour l'analyse. Exclure ces personnes biaiserait la distribution.

In [23]:
nb_participants_without_age = int(participants['age'].isna().astype(int).sum())
print(
    f"{nb_participants_without_age} / {nb_participants} " +
    "participants n'ont pas renseigné leur âge (i.e., " +
    f"{nb_participants_without_age / nb_participants:.1%}).",
    end="\n\n"
)

participants_age = participants[["age", "gender"]].copy()
participants_age["gender"] = participants_age["gender"].map(GENDER_LABELS)
age_stats = participants_age.groupby("gender").describe().round(1)
print(age_stats.to_string(), end="\n\n")

fig = go.Figure()
for gender in participants_age["gender"].unique():
    subset = participants_age[participants_age["gender"] == gender]
    fig.add_trace(
        go.Box(
            x=subset["age"],
            y=subset["gender"],
            name=gender,
            orientation="h",
            marker=dict(color=GENDER_COLORS_FROM_LABELS[gender])

    )
)
fig.update_layout(
    title='Répartition des âges par genre',
    #width=1200,
    height=400,
)

fig.update_xaxes(range=[0, round(get_column_min_max(participants_age, "age")[1], 0) + 5])
fig.update_traces(xhoverformat=".1f")
fig.show()

8 / 551 participants n'ont pas renseigné leur âge (i.e., 1.5%).

          age                                         
        count  mean  std   min   25%   50%   75%   max
gender                                                
Femmes  269.0  26.1  4.0  19.0  23.0  26.0  28.0  55.0
Hommes  274.0  26.6  3.5  18.0  24.0  27.0  29.0  42.0



### Domaines d'études

Le plus grand domaine d'études représenté est, avec près d'un quart de la population (23,6%), celui regroupant les affaires, l'économie et la finance.
Il est environ deux fois plus représenté que le second domaine regroupant la biologie, la chimie et la physique.
L'ingénierie est troisième avec 10,2%.
En ajoutant le quatrième groupe désignant le droit (8,7%), sur les 17 possibles, la majorité de la population est déjà caractérisée.

In [24]:
FIELD_LABELS = {
    1: 'Droit',
    2: 'Mathématiques',
    3: 'Sciences sociales, Psychologie',
    4: 'Sciences médicales, pharmaceutiques et Bio Tech',
    5: 'Ingénierie',
    6: 'Anglais / Écriture créative / Journalisme',
    7: 'Histoire / Religion / Philosophie',
    8: 'Affaires / Économie / Finance',
    9: 'Éducation',
    10: 'Sciences biologiques / Chimie / Physique',
    11: 'Travail social',
    12: 'Étudiant en premier cycle / non décidé',
    13: 'Sciences politiques / Affaires internationales',
    14: 'Cinéma',
    15: 'Beaux-arts / Gestion culturelle',
    16: 'Langues',
    17: 'Architecture',
    18: 'Autre',
    np.nan: 'Valeur manquante'
}

fields = participants['field_cd'].map(FIELD_LABELS).value_counts().sort_values().reset_index()
fields.columns = ['field', 'count']
fields['count_pct'] = fields['count'] / fields['count'].sum() * 100

colors = [ATTENTION_PLOT_COLOR if field == "Missing value" else DEFAULT_PLOT_COLOR for field in fields['field']]

fig = go.Figure(
    go.Bar(
        x = fields['count_pct'],
        y = fields['field'],
        orientation = 'h',
        marker = dict(color = colors)
    )
)
fig.update_layout(
    title="Domaine d'études des participants",
    height=500,
    showlegend=False,
    xaxis_title="Nombre de participants (%)",
    yaxis_title="Domaine"
)
fig.show()

### Objectif de participation

Concernant l'objectif de participation à la soirée de speed dating, deux sont clairement sur-représentés, quel que soit le genre. Le premier est "passer une bonne soirée" (42% pour les femmes et 40,8% pour les hommes) et le deuxième est "rencontrer de nouvelles personnes" (femmes 36,5%, hommes 32,1%). Les quatre autres objectifs sont tous approximativement dans l'intervalle [5, 10]%.
Nous pouvons remarquer que les deux genres répondent de manière quasi similaire à cette question.

In [25]:
GOAL_LABELS = {
    1: 'Passer une bonne soirée',
    2: 'Rencontrer de nouvelles personnes',
    3: 'Avoir un date',
    4: 'Chercher une relation sérieuse',
    5: "Dire que je l'ai fait",
    6: 'Autre',
    np.nan: 'Valeur manquante'
}

goals = participants[["gender", "goal"]].copy()
goals["gender"] = goals["gender"].map(GENDER_LABELS)
goals["goal"] = goals["goal"].map(GOAL_LABELS)

goals_ct = (100 * pd.crosstab(goals['goal'], goals['gender'], normalize="columns")).round(1)
goals_ordered = [wrap_label(goal, max_chars=15) for goal in goals_ct.sum(axis=1).sort_values(ascending=False).index.tolist()]
goals_lg = goals_ct.reset_index().melt(id_vars='goal', var_name='Gender', value_name='Nb_participants_pct')
goals_lg['goal'] = goals_lg['goal'].apply(wrap_label, max_chars=15)

fig = px.bar(
    goals_lg,
    x='goal',
    y='Nb_participants_pct',
    category_orders={"goal": goals_ordered},
    color='Gender',
    barmode='group',
    color_discrete_map=GENDER_COLORS_FROM_LABELS,
    title="Raison donnée pour la participation à la soirée",
    labels={'Nb_participants_pct': 'Nombre de participants (%)', 'goal': 'Objectif'}
)
fig.update_layout()
fig.show()

### Importances du partage de l'origine et de la religion

Les graphes ci-dessous montrent qu'une grande partie de la population, hommes et femmes confondus, n'attache pas beaucoup d'importance au partage de l'origine et de la religion avec leur partenaire.
Par ailleurs, les deux graphes suivent les mêmes tendances :
- Entre globalement un quart et la moitié de la population considère ces deux facteurs comme pas du tout importants (les hommes marquent encore plus ce manque d'importance).
- La densité de population décroît globalement avec l'importance donnée.
- Les distributions des deux importances sont très similaires chez les hommes et les femmes.

In [26]:
imp_race_relig = participants[['gender', 'imprace', 'imprelig']]
imp_race_relig["gender"] = imp_race_relig["gender"].map(GENDER_LABELS)

imp_race_relig_NA = imp_race_relig[["imprace", "imprelig"]].isna().sum(axis=1).value_counts().reset_index()
imp_race_relig_NA.rename(columns={"index": "Nombre de donnés manquantes", "count": "Nombre de participants (-)"}, inplace=True)
imp_race_relig_NA["Commentaires"] = imp_race_relig_NA["Nombre de donnés manquantes"].map({0: "Pas de données manquantes", 1: "Une donnée manquante", 2: "Deux données manquantes"})
print(imp_race_relig_NA.to_string(index=False), end="\n\n")
print(
    "Aucun participant n'a répondu qu'à une seule des deux questions sur " +
    "l'importance d'avoir la même origine et la même religion.\n" +
    "Donc les lignes avec des valeurs manquantes peuvent simplement être " +
    "supprimées pour cette analyse.\n"
)
imp_race_relig = imp_race_relig.dropna(how="any")

fig = make_subplots(
    cols=2,
    column_titles=["Importance d'avoir la même origine", "Importance d'avoir la même religion"]
)

for gender in imp_race_relig['gender'].unique():
    subset = imp_race_relig[imp_race_relig['gender'] == gender]

    fig.add_trace(
        go.Histogram(
            x=subset["imprace"],
            histnorm='percent',
            name=gender,
            marker=dict(color=GENDER_COLORS_FROM_LABELS[gender]),
            legendgroup=gender
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Histogram(
            x=subset["imprelig"],
            histnorm='percent',
            name=gender,
            marker=dict(color=GENDER_COLORS_FROM_LABELS[gender]),
            legendgroup=gender,
            showlegend=False
        ),
        row=1, col=2
    )
fig.update_xaxes(range=[0, 11], tickvals=list(range(1, 11)))
for i in range(2):
    fig.update_xaxes(
        title="Importance",
        row=1,
        col=i + 1
    )
    fig.update_yaxes(
        title="Nombre de participants (%)",
        row=1,
        col=i + 1
    )
fig.update_layout(
    height=400,
    barmode="group",
)

fig.show()

 Nombre de donnés manquantes  Nombre de participants (-)              Commentaires
                           0                         544 Pas de données manquantes
                           2                           7   Deux données manquantes

Aucun participant n'a répondu qu'à une seule des deux questions sur l'importance d'avoir la même origine et la même religion.
Donc les lignes avec des valeurs manquantes peuvent simplement être supprimées pour cette analyse.



### Bilan partiel — Partie 1

- Population équilibrée : 274 femmes et 277 hommes, âgés d'environ 26 ans en moyenne
- Profil dominant : étudiants en business/finance (23,6 %), sciences, ingénierie
- Objectifs majoritaires : "passer une bonne soirée" et "rencontrer de nouvelles personnes" (cumul des deux > 70 %)
- Origine et religion partagées : faible importance déclarée, tous genres confondus
- Décisions positives : 42 % en moyenne (hommes 47,4 % vs femmes 36,5 %)
- Taux de match : 16,5 %, soit 1 match sur 6 speed dates

---

## Partie 2 — Facteurs déclarés pour l'attirance

### Ce que recherchent les candidats chez un partenaire

Rappel : les candidats établissent leurs préférences à l'inscription, puis à d'autres instants.

Le graphe ci-dessous montre les préférences réparties sur les six attributs définis en introduction, différenciées par genre.
Nous pouvons noter une différence notable entre les hommes et les femmes : les hommes indiquent clairement rechercher une attractivité physique (27,2%), attribut le plus hautement noté tous genres confondus, et bien démarqué du deuxième qui est l'intelligence (19,4%). Les deux attributs suivants, fun (17,5%) et sincérité (16,3%), sont proches. Les deux derniers sont nettement plus faiblement notés avec en cinquième les intérêts partagés (10,9%) et en dernier l'ambition (8,7%).
Les femmes ont une distribution des attributs plus équilibrée. Elles recherchent en premier un partenaire intelligent (21,0%), sincère (18,2%), physiquement attirant (18,0%), fun (17,3%), ambitieux (12,8%) et, en dernier, avec des intérêts partagés (12,7%).

Résumé des préférences :

| Ordre de préférence    | Attribut choisi par les femmes | Attribut choisi par les hommes |
|------------------------|--------------------------------|--------------------------------|
| 1 (le plus important)  | Intelligence                   | Attirance physique             |
| 2                      | Sincérité                      | Intelligence                   |
| 3                      | Attirance physique             | Fun                            |
| 4                      | Fun                            | Sincérité                      |
| 5                      | Ambition                       | Intérêts partagés              |
| 6 (le moins important) | Intérêts partagés              | Ambition                       |


In [27]:
# Ce qu'un candidat veut chez un partenaire
preferences_self = participants[["gender"] + find_column_names(participants, startswith=ATTR_ABBREVS, endswith=["1_1"])]
# Ce qu'un candidat pense que les autres candidats du même sexe veulent chez un partenaire
preferences_peers = participants[["gender"] + find_column_names(participants, startswith=ATTR_ABBREVS, endswith=["4_1"])]
# Ce qu'un candidat pense que les candidats du sexe opposé veulent chez un partenaire
preferences_opposite_sex = participants[["gender"] + find_column_names(participants, startswith=ATTR_ABBREVS, endswith=["2_1"])]

preferences_self.columns = ["gender"] + ATTR_ABBREVS
preferences_peers.columns = ["gender"] + ATTR_ABBREVS
preferences_opposite_sex.columns = ["gender"] + ATTR_ABBREVS

preferences_self_avg_by_gender = preferences_self.groupby('gender')[ATTR_ABBREVS].mean().rename(index=GENDER_LABELS)

preferences_self_avg_by_gender_lg = preferences_self_avg_by_gender.reset_index().melt(
    id_vars='gender', var_name='attribute', value_name='pct'
)
preferences_self_avg_by_gender_lg['Attribut'] = preferences_self_avg_by_gender_lg['attribute'].map(ATTR_LABELS)
preferences_self_avg_by_gender_lg.rename(columns={'gender': 'Genre'}, inplace=True)

fig = px.bar(
    preferences_self_avg_by_gender_lg,
    x='Attribut',
    y='pct',
    color='Genre',
    barmode='group',
    color_discrete_map=GENDER_COLORS_FROM_LABELS,
    title="Ce qu'un candidat recherche chez son partenaire",
    labels={'pct': 'Répartition moyenne des préférences (%)'}
)
fig.add_hline(
    y=100/len(ATTR_ABBREVS),
    line_dash='dash',
    line_color='gray',
    annotation_text=f"Ligne d'équirépartition ({100/len(ATTR_ABBREVS):.1f}%)",
    annotation_position='top right'
)
fig.show()

### Confrontation : candidat vs. sexe opposé

Il est intéressant de confronter ces deux visions des préférences (déclarées à l'inscription) :
- ce que le candidat pense que le sexe opposé recherche chez un partenaire
- ce que le sexe opposé dit qu'il recherche chez un partenaire (cf. graphe précédent).

Avec ce genre de confrontation, nous pouvons espérer déceler des biais que les candidats ont sur le sexe opposé ou sur eux-mêmes.

Sur le graphe ci-dessous, nous notons que les femmes connaissent le premier critère chez les hommes, c'est-à-dire l'attirance physique, mais qu'elles surévaluent largement ce critère par rapport à ce que les hommes disent de ce critère (+8,3%). A contrario, elles sous-évaluent l'importance de l'intelligence (-6,9%) et de la sincérité (-3,0%). Les trois critères restants sont mieux évalués (différences inférieures à 2%).

Les hommes surévaluent également l'importance de l'attirance physique (+7,0%), sous-évaluent aussi l'intelligence (-4,6%) et la sincérité (-3,1%) et estiment précisément les trois autres critères (à moins de 2% près).

In [28]:
# Confrontation :
labels = [
    "Ce qu'un candidat pense que les candidats du sexe opposé veulent chez un partenaire",
    "Ce que les candidats du sexe opposé veulent vraiment chez un partenaire"
]
labels = [wrap_label(lbl, max_chars=50) for lbl in labels]

preferences_NA_indices = {
    labels[0]: preferences_opposite_sex,
    labels[1]: preferences_self
}
preferences_NA = pd.DataFrame(
    [
        {k: v.drop(columns=['gender']).isna().any(axis=1).sum() for k, v in preferences_NA_indices.items()}
    ]
)

print(
    (
        preferences_NA
        .transpose()
        .rename(columns={0: "Nombre de lignes avec des données manquantes"})
        .to_string()
        .replace("<br>", " ")
    ),
    end="\n\n"
)

preferences_opposite_sex_avg_prefs = preferences_opposite_sex.groupby('gender')[ATTR_ABBREVS].mean()
preferences_self_avg_prefs = preferences_self.groupby('gender')[ATTR_ABBREVS].mean()
records = []
for i in range(2):
    label_ = labels[i]
    for gender_id in [0, 1]:
        for attr in ATTR_ABBREVS:
            if i == 0:
                avg_prefs = preferences_opposite_sex_avg_prefs.copy().iloc[gender_id]
            else:
                avg_prefs = preferences_self_avg_prefs.copy().iloc[1 - gender_id]
            records.append(
                {
                    'Genre': GENDER_LABELS[gender_id],
                    'Attribut': wrap_label(ATTR_LABELS[attr], max_chars=15),
                    'pct': avg_prefs.iloc[ATTR_ABBREVS.index(attr)],
                    'Vue': label_
                }
            )

confronted_results = pd.DataFrame(records)

fig = px.bar(
    confronted_results,
    x='Attribut',
    y='pct',
    color='Vue',
    facet_col='Genre',
    barmode='group',
    color_discrete_sequence=[ATTENTION_PLOT_COLOR, DEFAULT_PLOT_COLOR],
    title="Estimation des préférences du sexe opposé par un candidat vs préférences réelles",
    labels={'pct': 'Valeur moyenne (%)'}
)
titles = {f"Genre={gender}": gender for gender in GENDER_LABELS.values()}
fig.for_each_annotation(
    lambda a: a.update(
        text=titles.get(a.text, a.text),
        y=a.y - 0.05
    )
)
fig.update_layout(
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.25,
        xanchor='center',
        x=0.5
    )
)
fig.show()

                                                                                        Nombre de lignes avec des données manquantes
Ce qu'un candidat pense que les candidats du sexe opposé veulent chez un partenaire                                             7
Ce que les candidats du sexe opposé veulent vraiment chez un partenaire                                                         7



### Bilan partiel — Partie 2

- Hommes : **attirance physique** clairement en tête (27,2 %).
- Femmes : préférences plus équilibrées, **intelligence** en tête (21,0 %).
- Les deux genres **surestiment** l'importance de l'attirance physique pour le sexe opposé (+7 à +8 %).
- Les deux genres **sous-estiment** l'importance de l'intelligence et de la sincérité.
- Premier écart visible entre **représentations** et **réalité déclarée par l'autre**.

---

## Partie 3 — Facteurs réels pour l'attirance

Après la partie 2 de l'étude où nous nous sommes intéressés à ce que les candidats établissent comme préférences pour eux et ce qu'ils pensent des préférences du sexe opposé, explorons à présent les attributs qui les font réellement dire "oui" à leur partenaire.

Pour cela, nous prenons comme données, pour chaque speed date, les décisions rendues pour chaque candidat ('oui' ou 'non'), s'il y a match ou non, et les notes qu'il a données à son partenaire.
Nous calculons alors les corrélations :
- entre ces attributs et la décision,
- entre ces attributs et le match

Les résultats sont affichés sur les graphes ci-dessous.

Nous notons que les préférences réelles des femmes et des hommes sont assez proches, que ce soit pour rendre un 'oui' ou pour un match :
- Les amplitudes sont quasi identiques (3,3% d'écart au maximum, pour l'attirance physique, sur la décision).
- Pour le 'oui', l'ordre d'importance est le même pour les trois premiers critères, puis les trois derniers critères sont dans un autre ordre, mais tous avec des amplitudes très rapprochées (entre 9,2% et 12,2%). Les ordres sont indiqués dans le tableau ci-dessous.
- Pour le match, le "top 3" des critères (attirance physique, fun, intérêts partagés) et le "flop 3" (sincérité, intelligence, ambition) sont identiques à ceux obtenus pour le 'oui', mais avec des variations des classements à l'intérieur de ces triplets. Il est intéressant de noter que l'attirance physique n'est plus le premier critère pour le match et passe, en moyenne sur les deux genres, après le fun et les intérêts partagés, dans cet ordre. L'ordre du "flop 3" est inchangé. Nous remarquons que le nivellement entre les six attributs tend à se resserrer.

| Ordre de préférence    | Femmes - 'oui'     | Hommes - 'oui'     | Femmes - match     | Hommes - match     |
|------------------------|--------------------|--------------------|--------------------|--------------------|
| 1 (le plus important)  | Attirance physique | Attirance physique | Fun                | Attirance physique |
| 2                      | Fun                | Fun                | Intérêts partagés  | Fun                |
| 3                      | Intérêts partagés  | Intérêts partagés  | Attirance physique | Intérêts partagés  |
| 4                      | Intelligence       | Ambition           | Intelligence       | Intelligence       |
| 5                      | Sincérité          | Intelligence       | Sincérité          | Sincérité          |
| 6 (le moins important) | Ambition           | Sincérité          | Ambition           | Ambition           |

In [29]:
# Correlation entre la décision et chaque note d'attribut donné à un partenaire
raw_corr_dec_attrs = pd.DataFrame(
    {
        GENDER_LABELS[g]: df[df['gender'] == g][['dec'] + ATTR_ABBREVS].corr()['dec'].drop('dec')
        for g in GENDER_LABELS.keys()
    }
)
corr_dec_attrs = pd.DataFrame(
    {
        g: {attr: 100 * corr[attr] / corr.sum() for attr in ATTR_ABBREVS}
        for g, corr in raw_corr_dec_attrs.items()
    }
)

corr_dec_attrs_lg = corr_dec_attrs.loc[ATTR_ABBREVS].reset_index().melt(
    id_vars='index', var_name='Genre', value_name='pct'
)
corr_dec_attrs_lg['Attribut'] = corr_dec_attrs_lg['index'].map(ATTR_LABELS)
corr_dec_attrs_lg['variable'] = "dec"


# Correlation entre le match et chaque note d'attribut donné à un partenaire
raw_corr_match_attrs = pd.DataFrame(
    {
        GENDER_LABELS[g]: df[df['gender'] == g][['match'] + ATTR_ABBREVS].corr()['match'].drop('match')
        for g in GENDER_LABELS.keys()
    }
)
corr_match_attrs = pd.DataFrame(
    {
        g: {attr: 100 * corr[attr] / corr.sum() for attr in ATTR_ABBREVS}
        for g, corr in raw_corr_match_attrs.items()
    }
)

corr_match_attrs_lg = corr_match_attrs.loc[ATTR_ABBREVS].reset_index().melt(
    id_vars='index', var_name='Genre', value_name='pct'
)
corr_match_attrs_lg['Attribut'] = corr_match_attrs_lg['index'].map(ATTR_LABELS)
corr_match_attrs_lg['variable'] = "match"

corr_dec_and_match_attrs_lg = pd.concat([corr_dec_attrs_lg, corr_match_attrs_lg], axis=0, ignore_index=True).drop(columns='index')

for var in ['dec', 'match']:
    print(f"Classement des attributs pour '{var}':\n" +
          (
              corr_dec_and_match_attrs_lg
              [corr_dec_and_match_attrs_lg["variable"] == var]
              [["Genre", "Attribut", "pct"]]
              .groupby("Attribut")
              ["pct"]
              .mean()
              .sort_values(ascending=False)
              .round(1)
              .to_string()
          ),
          end="\n\n"
          )

fig = px.bar(
    corr_dec_and_match_attrs_lg,
    x='Attribut',
    y='pct',
    color='Genre',
    barmode='group',
    facet_row='variable',
    color_discrete_map=GENDER_COLORS_FROM_LABELS,
    title="Corrélation normalisée entre les attributs et le 'oui' / le match par genre",
    labels={'pct': "Correlation normalisée (%)"}
)
titles = {f"variable={k}": v for k, v in {"dec": "Décision 'Oui'", "match": "Match"}.items()}
fig.for_each_annotation(
    lambda a: a.update(
        text=titles.get(a.text, a.text),
        x=0.5,
        xanchor='center',
        yanchor='bottom',
        y=a.y + 0.2,
        textangle=0
    )
)
fig.update_layout(
    height=800,
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.1,
        xanchor='center',
        x=0.5
    )
)
fig.update_xaxes(showticklabels=True)
fig.update_traces(yhoverformat=".1f")
fig.show()


Classement des attributs pour 'dec':
Attribut
Attirance physique    25.0
Fun                   21.4
Intérêts partagés     20.9
Intelligence          11.7
Sincérité             10.8
Ambition              10.3

Classement des attributs pour 'match':
Attribut
Fun                   21.5
Intérêts partagés     21.0
Attirance physique    20.5
Intelligence          13.2
Sincérité             12.8
Ambition              11.1



Le graphe suivant permet de comparer les préférences déclarées par les participants à l'inscription avec celles qui sont corrélées à leur décision de dire "oui" pour un second date à leur partenaire.

Concernant les femmes, nous notons **un bouleversement entre l'ordre déclaré et l'ordre réel**. L'intelligence n'est pas du tout le premier attribut (en l'occurrence le quatrième). En effet, l'attirance physique tient la première place. Le deuxième est le fun (déclaré quatrième), presque à égalité avec les intérêts partagés (déclarés derniers). L'intelligence et la sincérité, plutôt déclarées de premier rang (premier et deuxième), sont en réalité d'une bien moindre importance (quatrième et cinquième). Quant à l'ambition, déclarée avant-dernière, confirme son importance mineure et est dernière.

Pour les hommes, la prépondérance de l'attirance physique est confirmée (première place). Ensuite, un gros changement, similaire à celui des femmes s'opère. Le fun (déclaré troisième) est deuxième, les intérêts partagés (déclarés avant-derniers) suivent, l'ambition (déclarée dernière) est quatrième et, finalement, l'intelligence et la sincérité (respectivement deuxième et quatrième) occupent les deux dernières places.

En résumé, pour qu'un candidat donne son "oui", les critères jouant un **rôle majeur** sont, dans cet ordre : **attirance physique**, **fun** et **intérêts partagés**.
À l'inverse, l'intelligence, la sincérité et l'ambition sont des critères qui ne permettent pas la mise en valeur d'un partenaire.

In [30]:
# Stated vs. real preferences
# - Stated: mean of preferences at signup (% of total)
# - Real: weight in the decision, using correlations as a measure of real importance

stated_prefs = preferences_self_avg_by_gender_lg.drop(columns="attribute").sort_values("Genre").reset_index()
real_prefs = corr_dec_attrs_lg.drop(columns="index").reset_index()
stated_prefs["Type de préférence"] = "Préférences déclarées"
real_prefs["Type de préférence"] = "Préférences réelles"
stated_vs_real_prefs = pd.concat([stated_prefs, real_prefs], ignore_index=True, axis=0)
stated_vs_real_prefs["Attribut"] = stated_vs_real_prefs["Attribut"].apply(wrap_label, max_chars=15)

fig = px.bar(
    stated_vs_real_prefs,
    x='Attribut',
    y='pct',
    color="Type de préférence",
    facet_col='Genre',
    barmode='group',
    color_discrete_sequence=[ATTENTION_PLOT_COLOR, DEFAULT_PLOT_COLOR],
    title="Préférences déclarées vs préférences réelles",
    labels={'pct': 'Importance (%)'}
)
titles = {f"Genre={gender}": gender for gender in GENDER_LABELS.values()}
fig.for_each_annotation(
    lambda a: a.update(
        text=titles.get(a.text, a.text),
        y=a.y - 0.05
    )
)
fig.update_layout(
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.2,
        xanchor='center',
        x=0.5
    )
)
fig.update_layout(height=400)
fig.update_traces(yhoverformat=".1f")
fig.update_yaxes(showticklabels=True)
fig.show()

### Bilan partiel — Partie 3

- Préférences **réelles quasi identiques entre les deux genres**
- Trio gagnant pour le "oui" : **attirance physique, fun, intérêts partagés**
- Pour le **match** : le fun passe devant l'attirance physique
- Trio perdant systématique : sincérité, intelligence, ambition
- **Bouleversement majeur** entre préférences déclarées et préférences réelles, particulièrement chez les femmes (l'intelligence chute du 1ᵉʳ au 4ᵉ rang)

---

## Partie 4 — Facteurs contextuels pour l'attirance

Dans cette partie, nous analysons l'effet de certains facteurs sur le taux de match :
- l'effet d'avoir des centres d'intérêt corrélés, mis en perspective avec le partage ou non de la même origine que le partenaire,
- l'effet de la position relative du speed date dans la soirée (plutôt au début, à la fin, etc.) sur l'obtention d'un second date,
- l'effet d'avoir un nombre de partenaires dans la soirée plutôt limité ou étendu
- et l'effet de la différence d'âge.

### Effet de la corrélation entre centres d'intérêt, perspective avec le partage de l'origine

Nous nous sommes servi de la colonne `int_corr` définissant la corrélation entre les intérêts du partenaire et du candidat déclarés au moment de l'inscription.
Nous avons regroupé les speed dates (c'est-à-dire les lignes du dataset) en fonction des valeurs de `int_corr` en 10 intervalles, sensiblement égaux en taille, que nous appellerons déciles, par abus de langage, et que nous nommons D1, ..., D10.

La sortie ci-dessous montre que les D1 et D2 regroupent presque la totalité des corrélations négatives, indiquant que cette partie mineure de la population possède des intérêts "contraires" avec le reste.
Elle montre aussi que D9 et D10 désignent des taux supérieurs à ~0,5.
Ensuite, les taux de match moyens relatifs au partage de l'origine sont indiqués : 16,0% si les partenaires sont d'origines différentes et 17,1% s'ils sont de la même origine.
La différence est mince et indique une légère préférence de chercher un partenaire de même origine.
Ce résultat est, pour partie, en contradiction avec ce que nous avions analysé dans la partie 1.
En effet, 30% des femmes et 40% des hommes déclarent ne porter aucune importance (note de 1/10) sur le partage de l'origine et environ la moitié de la population déclare au plus une importance assez faible (2 à 3/10).
Ceci pourrait simplement indiquer une différence entre ce que les candidats déclarent et leurs préférences réelles.

Le graphique ci-dessous affiche le taux moyen de match pour chaque décile. Deux lignes horizontales sont ajoutées pour repérer les taux moyens de match des groupes "origines différentes" et "mêmes origines". La tendance globale indique que, plus les partenaires ont des intérêts corrélés, plus il y a de chance d'avoir un match entre eux. Cependant, une zone, couverte par D2 et D3, atténue cette tendance et montre que deux partenaires avec des intérêts assez décorrélés (corrélation entre -0,21 et +0,03) ont un taux de match moyen (16,4%) équivalent à la moyenne obtenue sur tout le dataset (16,5%) et environ à mi-chemin entre les deux limites ajoutées sur le graphique.
D1 possède le taux le plus faible, égal à 13,7%, soit 2,3% de moins que le groupe "origines différentes" et 2,8% de moins que la moyenne.
À partir du décile D6, les taux moyens de match par décile dépassent celui du groupe "origines différentes" et l'écart croît strictement.
À partir du décile D9 (corrélation > 0,49), ces taux sont supérieurs au groupe "même origine" et atteignent le maximum en D10 (corrélation > 0,59) avec 20,0%, c'est-à-dire 3,5% de plus que la moyenne générale.

En résumé, partager la même origine que son partenaire augmente légèrement la chance d'obtenir un match (+1,1% à 17,1% en absolu, par rapport au groupe "origines différentes").
En revanche, partager les mêmes intérêts peut permettre d'augmenter davantage ses chances. Avoir une corrélation modérément élevée d'au moins 0,59 fait monter la chance d'avoir un match à 20%.

Pour poursuivre cette analyse, il serait intéressant d'étudier la dépendance du taux de match aux effets composés des intérêts et de l'origine (non effectué ici).

In [31]:
# Comparaison de deux effets sur le taux de match : corrélation des intérêts et partage de l'origine

n_bins = 10  # Nombre de bins (quantiles) pour la corrélation des intérêts

map_races = {0: 'Origine différente', 1:'Même origine'}

int_corr_and_samerace = df[['int_corr', 'samerace', 'match']].dropna().copy()
int_corr_and_samerace['int_corr_bin'] = pd.qcut(
    int_corr_and_samerace['int_corr'],
    q=n_bins,
    labels=[str(i) for i in range(1, n_bins + 1)]
)

print(
    "Nombre d'observations utilisées : " +
    f"{int_corr_and_samerace.shape[0]}" +
    f" ({int_corr_and_samerace.shape[0] / nb_rows:.1%})"
)
print(
    "Nombre d'observations supprimées : " +
    f"{nb_rows - int_corr_and_samerace.shape[0]}" +
    f" ({(nb_rows - int_corr_and_samerace.shape[0]) / nb_rows:.1%})"
)

print(
    f"\nInformation sur les quantiles (min, max, taille) et le taux de match moyen associé :\n" +
    (
        int_corr_and_samerace
        .groupby('int_corr_bin')
        .agg(
            int_corr_min=("int_corr", "min"),
            int_corr_max=("int_corr", "max"),
            int_corr_size=("int_corr", "count"),
            match_rate=("match", lambda x: (100 * x).mean().round(1)),
        )
        .to_string()
    ),
    end="\n\n"
)

print(
    f"\nTaux de match moyen par valeur de 'samerace' :\n" +
    (
        int_corr_and_samerace
        .groupby('samerace')
        ["match"]
        .apply(lambda x: (100 * x).mean().round(1))
        .rename(index=map_races)
        .to_string()
    ),
    end="\n\n"
)

effect_interests = int_corr_and_samerace.groupby('int_corr_bin', observed=True)['match'].mean()

effect_race = int_corr_and_samerace.groupby('samerace')['match'].mean()
effect_race.index = map_races.values()

fig = px.bar(
    effect_interests.reset_index(),
    x = "int_corr_bin",
    y = "match",
    title = "Taux de match par quantile de corrélation d'intérêts",
    color_discrete_sequence=[ALTERNATIVE_PLOT_COLOR],
    labels = {
        'int_corr_bin': f"Quantile de corrélation d'intérêts (1=la plus faible, {n_bins}=la plus élevée)",
        'match': 'Taux de match (%)'
    }
)
race_colors = {0: DEFAULT_PLOT_COLOR, 1: ATTENTION_PLOT_COLOR}
race_annotation_positions = {0: "bottom right", 1: "top left"}
race_alt_text = {0: "une origine différente", 1: "une même origine"}
for race in [0, 1]:
    fig.add_hline(
        y=effect_race.values[race],
        line_dash='dash',
        line_color=race_colors[race],
        annotation_text=f'Taux de match moyen pour {race_alt_text[race]} ({effect_race.values[race]:.1%})',
        annotation_position=f'{race_annotation_positions[race]}'
    )
fig.update_yaxes(tickformat=".0%")
fig.update_traces(yhoverformat=".1%")
fig.show()

Nombre d'observations utilisées : 8220 (98.1%)
Nombre d'observations supprimées : 158 (1.9%)

Information sur les quantiles (min, max, taille) et le taux de match moyen associé :
              int_corr_min  int_corr_max  int_corr_size  match_rate
int_corr_bin                                                       
1                    -0.83         -0.22            846        13.7
2                    -0.21         -0.07            844        16.4
3                    -0.06          0.03            779        16.4
4                     0.04          0.13            939        14.9
5                     0.14          0.21            731        15.0
6                     0.22          0.30            810        16.3
7                     0.31          0.38            819        16.6
8                     0.39          0.48            896        16.7
9                     0.49          0.58            736        18.5
10                    0.59          0.91            820        20.0


Tau

### Effet de la position relative du speed date dans la soirée sur l'obtention d'un second date

Dans cette section, nous analysons l'effet de la position relative du speed date dans la soirée :
- dans un premier temps, sur la décision de dire "oui" à son partenaire et d'obtenir un match,
- dans un second temps, sur l'obtention d'un second date.

Le but est d'étudier un phénomène de fatigue et d'attention chez les candidats et d'analyser s'il y a des moments clés dans la soirée.

Nous normalisons la position de passage (`order`) avec le nombre de speed dates effectué par un candidat dans la soirée (`round`).
Cette étape est indispensable, car le nombre de rencontres par candidat et par soirée varie fortement (entre 5 et 22).
Ensuite, les speed dates sont regroupés en 10 intervalles de largeur identique (*bins*) en position relative : 1 pour [0 ; 0,1], 2 pour [0,1 ; 0,2], ..., 10 pour [0,9 ; 1].
À chaque groupe est associé le taux moyen de décision positive et de match.
Pour chaque intervalle, les deux genres sont aussi dissociés.
Un diagramme en barres est alors tracé où, pour chaque intervalle de position relative, les taux moyens sont représentés.

Comme déjà énoncé dans la partie 1, les hommes donnent plus de "oui" que les femmes, en moyenne à 47,3% contre 36,5%.
Ce taux est assez stable jusqu'au bin 7, oscillant entre 46,6% et 49,6%, puis tombe pour atteindre 43,1% au bin 9, avant d'atteindre son maximum au bin 10 à 52,7%.
Les femmes ne suivent pas la même tendance. Le maximum de 42,6% est atteint dès le début de soirée, au bin 1, puis il est relativement stable jusqu'à la fin, sauf au bin 7 où le taux atteint son minimum à 33,0%.
Le taux de match est, par essence, corrélé aux taux de décision positive pour chaque genre. Par conséquent, sa tendance globale suit ces deux taux.
Il est maximal au bin 1 (car les femmes disent beaucoup "oui") et quasi maximal au bin 10 (idem pour les hommes).
Il subit un creux dans le dernier tiers, dans les bins 7 à 9, avec un minimum de 12,2% atteint au bin 7.

Sans rentrer dans des expertises psychologiques ou comportementales qui nous échappent, nous pouvons imaginer les scénarii possibles ci-dessous.
- La fatigue s'installant, l'attention des hommes et des femmes finit par baisser dans le dernier tiers de la soirée,
- Les femmes sont peut-être plus enthousiastes au démarrage de la soirée (probablement avant le premier speed date), les poussant à donner beaucoup de réponses positives très tôt, puis à se lasser.
- Les hommes manifestent le plus de décisions positives en toute fin de soirée, comme s'ils ne voulaient pas manquer une opportunité (au risque que les candidates les déçoivent a posteriori).
- Le bin 6 est situé juste après la mi-soirée et, probablement, après une pause pendant laquelle tous les candidats remplissent les formulaires. Cela pourrait expliquer une valeur élevée de réponses positives chez les hommes, mais non présente chez les femmes.

In [32]:
df_order = df.dropna(subset=['gender', 'order', 'round', 'dec', 'match']).copy()
df_order['order_pct'] = df_order['order'] / df_order['round']
nb_bins = 10
df_order['order_bin'] = pd.cut(
    df_order['order_pct'], bins=nb_bins,
    labels=[f"{i + 1}" for i in range(nb_bins)]
)
df_order = df_order[["gender", "dec", "match", "order_bin"]]

df_order_dec = pd.DataFrame(
    {
        "Taux de décision positive donné par les " + GENDER_LABELS[g].lower() + "s": df_order[df_order['gender'] == g].groupby('order_bin')['dec'].mean()
        for g in GENDER_LABELS.keys()
    }
)
df_order_match = df_order.groupby('order_bin')['match'].mean()
df_order = pd.concat([df_order_dec, df_order_match], axis=1)

print((100 * df_order).describe().round(1).drop(index=["count", "25%", "75%"]).to_string())

df_order_lg = df_order.reset_index().melt(id_vars='order_bin', var_name='Quantité', value_name='pct')
df_order_lg["Quantité"] = df_order_lg["Quantité"].str.replace("match", "Taux de match")
df_order_lg["pct"] *= 100

fig = px.bar(
    df_order_lg,
    x='order_bin',
    y='pct',
    color='Quantité',
    barmode='group',
    color_discrete_sequence=list(GENDER_COLORS_FROM_LABELS.values()) + [DEFAULT_PLOT_COLOR],
    title="Taux de 'oui' / match par position relative du speed date dans la soirée",
    labels={
        'order_bin': 'Position relative du speed date dans la soirée (1 : début - 10 : fin)',
        'pct': 'Valeur moyenne du taux (%)'
    }
)
fig.update_layout(
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.2,
        xanchor='center',
        x=0.5
    )
)
fig.show()

      Taux de décision positive donné par les femmess  Taux de décision positive donné par les hommess  match
mean                                             36.5                                             47.3   16.3
std                                               2.7                                              2.7    2.6
min                                              33.0                                             43.1   12.2
50%                                              35.8                                             47.2   16.5
max                                              42.6                                             52.7   20.2


Afin d'étudier l'effet de la position relative du speed date au cours de la soirée sur l'obtention d'un second date (post soirée), nous avons appliqué une méthode similaire à la précédente.

Avant d'analyser les résultats, il est important de noter que les données nécessaires à l'étude (colonne `date_3`) de cet effet sont incomplètes pour 52,6% du dataset. Les causes de ce manque peuvent être, par exemple :
- Phénomène d'attrition : plus on s'éloigne de l'événement, moins il y a de répondants.
- Manque d'incitation à répondre : à la dernière étape de remplissage des formulaires, il n'y a plus de monnaie d'échange (par exemple, au temps 2, les candidats reçoivent leurs matchs s'ils répondent au questionnaire).
- Désintérêt : une personne n'ayant reçu aucun match sera moins motivée à remplir la dernière partie du questionnaire.
- Oubli : après presque un mois, peut-être sans relance, certaines personnes oublient simplement.
- Lassitude du questionnaire : les questions peuvent demander un effort particulier, comme les répartitions de 100 points en six catégories.

Lors de cette analyse, un nouveau non-respect de la documentation a été trouvé.
En effet, outre les valeurs manquantes, le décomptage des valeurs de `date_3` montre qu'elles sont dans `{0 ; 1}` et pas dans `{1 ; 2}`.
Nous avons fait l'hypothèse que 0 a été utilisé pour "non" et 1 pour "oui".

Le graphique que nous avons tracé ci-dessous montre que la position relative du speed date au cours de la soirée n'a quasiment aucune influence sur l'obtention d'un second date. En effet, le taux d'obtention d'un second date va de 36,0% à 39,2%, tous genres confondus, avec des moyennes pour les hommes et les femmes de, respectivement, 37,7% et 37,6%.

En particulier, pour répondre à la question précisément posée et suggérée par Jedha, le premier et dernier speed dates de la soirée ont des chances quasi identiques (38,3% vs 38,4%) de se convertir en obtention d'un second date.

In [33]:
print(
    "Dénombrement (%) pour la colonne 'date_3':\n" +
    "\n".join((100 * df["date_3"].value_counts(dropna=False, normalize=True)).round(1).to_string().split("\n")[1:])
)
print("⚠️ Les valeurs devraient être de 1 ou 2 selon la documentation.\n")

print("On supprimes les valeurs manquantes et on fait l'hypothèse que 'oui' est associé à la valeur 1 and 'non' à 0.\n")
order_vs_2nd_date = df[['gender', 'order', 'round', 'date_3']].dropna(how="any").copy()
order_vs_2nd_date['order_pct'] = order_vs_2nd_date['order'] / order_vs_2nd_date['round']
order_vs_2nd_date["had_a_2nd_date"] = order_vs_2nd_date["date_3"].astype(int)
nb_bins = 10
order_vs_2nd_date['order_bin'] = pd.cut(
    order_vs_2nd_date['order_pct'], bins=nb_bins,
    labels=[f"{i + 1}" for i in range(nb_bins)]
)
order_vs_2nd_date = order_vs_2nd_date[["gender", "had_a_2nd_date", "order_bin"]]

order_vs_2nd_date_grp = pd.DataFrame(
    {
        GENDER_LABELS[g]: (
                100 *
                order_vs_2nd_date[order_vs_2nd_date['gender'] == g]
                .groupby('order_bin')
                ['had_a_2nd_date']
                .mean()
        ).round(1)
        for g in GENDER_LABELS.keys()
    }
)

print(order_vs_2nd_date_grp.describe().round(1).drop(index=["count", "25%", "75%"]).to_string(), end="\n\n")

print(
    'Taux moyen (%) de "a eu un 2ème date" par intervalle:\n' +
    "\n".join(order_vs_2nd_date_grp.mean(axis=1).round(1).to_string().split("\n")[1:]),
    end="\n\n"
)

order_vs_2nd_date_lg = order_vs_2nd_date_grp.reset_index().melt(id_vars='order_bin', var_name='Genre', value_name='had_2nd_date_rate')

fig = px.bar(
    order_vs_2nd_date_lg,
    x='order_bin',
    y='had_2nd_date_rate',
    color='Genre',
    barmode='group',
    color_discrete_sequence=list(GENDER_COLORS_FROM_LABELS.values()),
    title="Taux d'obtention d'un second date par position relative dans la soirée",
    labels={
        'order_bin': 'Position relative dans la soirée (1 : début - 10 : fin)',
        'had_2nd_date_rate': 'Taux de "a eu un 2ème date" (%)'
    }
)
fig.update_layout(
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.2,
        xanchor='center',
        x=0.5
    )
)
fig.show()

Dénombrement (%) pour la colonne 'date_3':
NaN    52.6
0.0    29.6
1.0    17.9
⚠️ Les valeurs devraient être de 1 ou 2 selon la documentation.

On supprimes les valeurs manquantes et on fait l'hypothèse que 'oui' est associé à la valeur 1 and 'non' à 0.

      Femmes  Hommes
mean    37.7    37.6
std      0.9     1.0
min     36.0    36.5
50%     37.9    37.5
max     39.0    39.2

Taux moyen (%) de "a eu un 2ème date" par intervalle:
1     38.3
2     37.5
3     36.6
4     37.3
5     38.4
6     37.8
7     36.4
8     38.2
9     37.4
10    38.4



### Effet d'un choix limité de partenaires sur le taux de décision positive et de match

Dans cette section, nous analysons l'effet de la limitation du nombre de partenaires dans la soirée sur le taux de décision positive et de match.
Le *booléen* `condtn` indique si un candidat pouvait rencontrer au plus 10 partenaires (valeur 1, *choix limité*) ou strictement plus de 10 partenaires (valeur 2, *choix étendu*).

Il est à noter que les deux parties du dataset constituant chaque possibilité sont inégales : 17,1% pour le choix limité contre 82,9% pour le choix étendu.

Le graphique ci-dessous indique une propension à donner plus facilement une réponse positive et, par conséquent, obtenir un match, dans le cadre d'un choix limité.
Contraindre le choix fait monter le taux de décision positive de près de 5% et le taux de match de 4,5%, ce qui n'est pas négligeable.

In [34]:
condtn_stats = df.groupby('condtn').agg(
    n_dates_pct=('dec', lambda x: (100 * x.count() / nb_rows).round(1)),
    yes_rate=('dec', 'mean'),
    match_rate=('match', 'mean')
)
condtn_stats[['yes_rate', "match_rate"]] *= 100
condtn_stats.rename(columns={'n_dates_pct': 'Nombre de speed dates (%)', 'yes_rate': 'Taux de décision positive', 'match_rate': "Taux de match"}, inplace=True)
condtn_stats.index = ['Choix limité', 'Large choix']

print("Effet d'un choix limité:\n" + condtn_stats.round(1).to_string() + "\n")

condtn_stats_lg = condtn_stats.drop(columns="Nombre de speed dates (%)").reset_index().melt(id_vars='index', var_name='Métrique', value_name='rate')

fig = px.bar(
    condtn_stats_lg, x='index', y='rate', color='Métrique',
    barmode='group',
    color_discrete_map={'Taux de décision positive': DEFAULT_PLOT_COLOR, 'Taux de match': ATTENTION_PLOT_COLOR},
    title="Effet d'un choix limité sur les taux de décision positive reçue et de match",
    labels={"index": "Type de choix", 'rate': 'Taux (%)'}
)
fig.update_layout(width=600)
fig.show()


Effet d'un choix limité:
              Nombre de speed dates (%)  Taux de décision positive  Taux de match
Choix limité                       17.1                       46.0           20.2
Large choix                        82.9                       41.2           15.7



### Effet de la différence d'âge entre partenaires sur le taux de match

Dans cette section, nous analysons l'effet de la différence d'âge entre les partenaires sur le taux de match.
Nous la définissons comme étant la différence entre l'âge du candidat et de son partenaire et nous créons la variable correspondante (`age_diff`).
Ainsi, une différence positive indique qu'un candidat est plus âgé que son partenaire, et vice versa.

Nous groupons le dataset par les valeurs de `age_diff` (et du genre) et calculons la moyenne du taux de match associé à chaque groupe.
Les données transformées obtenues étant difficilement exploitables, nous les modifions au moyen d'un filtre LOWESS (*Locally Weighted Scatterplot Smoothing*),
permettant un lissage.

Il est à noter que, par construction, les courbes (initiales ou lissées) traçant le taux de match en fonction de la différence d'âge sont symétriques par rapport à l'axe Y pour les hommes et les femmes. Nous pourrions ainsi ne tracer qu'une seule des deux courbes pour interpréter les résultats. Nous avons toutefois décidé de laisser les deux.

Le taux de match est maximal quand l'homme est âgé d'un an de plus que la femme, à un an près, et se situe à 18%.
Il chute à 15% à +5 ans et -4 ans. Différentes zones de stagnation sont présentes sur les courbes, mais nous ne les commenterons pas.
Le point important à retenir est que les deux candidats ont plus de chance d'obtenir un match s'ils ont des âges suffisamment proches, avec un écart au maximum de cinq ans, et idéalement avec l'homme légèrement plus âgé (zéro à deux ans).

In [35]:
df_age = df.dropna(subset=['age', 'age_o', 'match']).copy()
df_age['age_diff'] = (df_age['age'] - df_age['age_o']).astype(int)
df_age = df_age[["gender", "age_diff", "match"]]
match_rate = df_age.groupby(['gender', 'age_diff'])['match'].mean().reset_index()

fig = go.Figure()
for gender in match_rate['gender'].unique():
    subset = match_rate[match_rate['gender'] == gender]
    smoothed = lowess(subset['match'], subset['age_diff'], frac=0.25)
    subset['age_diff'] = smoothed[:, 0]
    subset['match'] = smoothed[:, 1]
    fig.add_trace(
        go.Scatter(
            x=subset['age_diff'],
            y=subset['match'],
            name=GENDER_LABELS[gender],
            mode='lines',
            line=dict(
                color=GENDER_COLORS_FROM_LABELS[GENDER_LABELS[gender]],
                shape='spline', smoothing=1.3
            )
        )
    )
fig.update_layout(title="Taux de match vs différence d'âge par genre")
fig.update_xaxes(title="Différence d'âge (candidat - partenaire) [année]", range=[-20, 20])
fig.update_yaxes(title="Taux de match moyen [%]", tickformat=".0%", range=[0, 0.2])
fig.show()

### Bilan partiel — Partie 4

- **Intérêts partagés > même origine** : forte corrélation des intérêts (>0,59) → taux de match à 20 %
- Même origine : effet marginal (+1,1 % seulement)
- **Position du speed date dans la soirée** : impact quasi nul sur l'obtention d'un second date (~38 % partout)
- **Choix limité** (≤10 partenaires) : +5 % de "oui", +4,5 % de match
- **Différence d'âge** : optimum à 0–2 ans (homme légèrement plus âgé) ; dégradation nette au-delà de ±5 ans

---

## Partie 5 — Évaluation : auto-évaluation vs évaluations reçues par les partenaires

Dans cette partie, nous analysons la perception que les candidats ont de leur propre valeur sur le marché du dating en comparant :
- leur auto-évaluation de leurs attributs à celles reçues par leurs partenaires,
- leur estimation d'obtenir une décision positive de la part du partenaire à la décision réellement rendue par celui-ci.

### Évaluation des attributs

Pour chaque genre et chaque attribut (sauf les intérêts partagés), nous avons mis en perspective les notes que les candidats
se sont données avec les notes qu'ils ont reçues en moyenne par leurs partenaires.

Le graphique ci-dessous montre, pour chacun des deux genres et chacun des cinq critères, un nuage de points des notes auto-attribuées et également reçues.
Pour faciliter la lecture, une ligne en pointillé est ajoutée, démarquant la zone de sur-évaluation des candidats par eux-mêmes (triangle inférieur) de la zone de sous-évaluation (triangle supérieur).
Un marqueur (croix blanche) est aussi ajouté pour situer la moyenne du nuage de points.

Nous constatons que, en moyenne, les hommes et les femmes surestiment tous leurs attributs.
La moyenne des biais sur l'ensemble est de +11,5% pour les femmes et de +8,5% pour les hommes.
Cela voudrait dire que **la population se sur-évalue globalement d'un point sur chaque critère**.
Dans le détail, chaque genre surestime ses attributs différemment :
- Les femmes se voient vraiment plus fun (+15,9%), attirantes et sincères (+13,2% pour les deux) qu'elles ne le sont, selon les estimations reçues par les hommes.
- Les hommes se considèrent clairement plus intelligents (+11,6%) qu'ils ne sont perçus par les femmes. Ils s'auto-surestiment plus modérément sur les autres critères.

In [36]:
# For each participant, compute the average received rating on each attribute

attrs = [a for a in ATTR_ABBREVS if a != "shar"]
self_ratings_cols = {a: find_column_names(df, startswith=[a], endswith=["3_1"])[0] for a in attrs}
received_ratings_cols = {a: find_column_names(df, startswith=[a], endswith=["_o"])[0] for a in attrs}

received_ratings = df.groupby('pid')[list(received_ratings_cols.values())].mean()
received_ratings.columns = [c.replace('_o', '') for c in received_ratings.columns]
received_ratings.index.name = 'iid'

self_ratings = participants.set_index('iid')[list(self_ratings_cols.values())]
self_ratings.columns = [c.replace('3_1', '') for c in self_ratings.columns]

self_vs_received_ratings = self_ratings.join(received_ratings, lsuffix='_self', rsuffix='_received').dropna()
self_vs_received_ratings['gender'] = participants.set_index('iid').loc[self_vs_received_ratings.index, 'gender']

attrs_self = self_ratings.columns

bias_str = "Biais moyen (auto-évaluation - évaluations reçues)"
print(bias_str + " [rating 1-10]:")
biases = pd.DataFrame(
    {
        attr: self_vs_received_ratings[f'{attr}_self'] - self_vs_received_ratings[f'{attr}_received']
        for attr in attrs_self
    }
)
biases['gender'] = self_vs_received_ratings['gender'].map(GENDER_LABELS)
biases_grp = biases.groupby('gender').mean().round(2)
print(biases_grp.to_string(float_format=lambda x: f"{x:+.2f}"), end="\n\n")

print(bias_str + " [%]:")
print(biases_grp.div(10).to_string(float_format=lambda x: f"{x:+.1%}"), end="\n\n")

print(bias_str.replace("(", "global (") + " [%]:")
print(biases_grp.mean(axis=1).div(10).to_string(float_format=lambda x: f"{x:+.1%}"), end="\n\n")

self_vs_received_avg_ratings = self_vs_received_ratings.groupby("gender").mean()

fig = make_subplots(
    rows=2,
    cols=5,
    subplot_titles=[ATTR_LABELS[attr] for attr in attrs_self],
    shared_yaxes=True
)

for i, attr in enumerate(attrs_self, start=1):
    for g in [0, 1]:
        gender = GENDER_LABELS[g]
        sub = self_vs_received_ratings[self_vs_received_ratings['gender'] == g]
        fig.add_trace(
            go.Scatter(
                x=sub[f'{attr}_self'],
                y=sub[f'{attr}_received'],
                mode='markers',
                marker=dict(
                    color=GENDER_COLORS_FROM_LABELS[gender],
                    size=6
                ),
                name=GENDER_LABELS[g],
                legendgroup=GENDER_LABELS[g],
                showlegend=(i == 1)
            ),
            row=g+1,
            col=i
        )
        # Perfect calibration
        fig.add_trace(
            go.Scatter(
                x=[0, 10],
                y=[0, 10],
                mode='lines',
                line=dict(
                    color="white",
                    dash='dash',
                    width=1
                ),
                showlegend=(i == 1 and g == 1),
                name='Auto-évaluation parfaite',
                legendgroup='ref',
            ),
            row=g+1,
            col=i
        )
        # Average
        fig.add_trace(
            go.Scatter(
                x=[self_vs_received_avg_ratings.iloc[g][f'{attr}_self']],
                y=[self_vs_received_avg_ratings.iloc[g][f'{attr}_received']],
                mode='markers',
                marker=dict(
                    symbol='cross',
                    size=10,
                    color='white'
                ),
                showlegend=(i == 1 and g == 1),
                name='Moyenne',
                legendgroup='avg',
            ),
            row=g + 1,
            col=i
        )
fig.for_each_annotation(
    lambda a: a.update(
        y=a.y - 0.025
    )
)
fig.update_xaxes(range=[0, 11], title_text='Auto-évaluation')
fig.update_yaxes(range=[0, 11])
fig.update_yaxes(title_text='Évaluation moyenne reçue', col=1)
fig.update_layout(
    height=600,
    title_text="Auto-évaluation vs évaluation moyenne reçue",
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.2,
        xanchor='center',
        x=0.5
    )
)
fig.show()

Biais moyen (auto-évaluation - évaluations reçues) [rating 1-10]:
        attr  sinc  intel   fun   amb
gender                               
Femmes +1.32 +1.32  +0.84 +1.59 +0.66
Hommes +0.47 +0.82  +1.16 +0.94 +0.87

Biais moyen (auto-évaluation - évaluations reçues) [%]:
         attr   sinc  intel    fun   amb
gender                                  
Femmes +13.2% +13.2%  +8.4% +15.9% +6.6%
Hommes  +4.7%  +8.2% +11.6%  +9.4% +8.7%

Biais moyen global (auto-évaluation - évaluations reçues) [%]:
gender
Femmes   +11.5%
Hommes    +8.5%



### Estimation d'obtention d'un "oui" de la part de l'autre

Le graphique ci-dessous trace, pour chaque genre, la probabilité observée d'obtenir une décision positive
de la part d'un partenaire en fonction de la probabilité estimée par le candidat.
Une ligne en pointillé partage l'espace en deux zones : un triangle inférieur indiquant la zone de "sur-confiance" pour les candidats et un triangle supérieur la zone de "sous-confiance".
Nous notons que, pour les deux genres, les courbes sont majoritairement situées dans la zone de "sur-confiance".
Cette tendance est encore plus marquée chez les hommes (courbe bleue plus éloignée de la ligne en pointillé).
Par exemple, quand les hommes sont confiants à 100%, ils n'obtiennent en moyenne en retour que 59 "oui" sur 100, alors que les femmes en reçoivent 76 pour la même confiance.

In [37]:
# Calibration of the 'prob' estimate (probability the other says yes) vs. reality (dec_o)

prob_vs_dec_o = df.dropna(subset=['gender', 'prob', 'dec_o']).copy()
prob_vs_dec_o = prob_vs_dec_o[['gender', 'prob', 'dec_o']]

prob_vs_dec_o = prob_vs_dec_o.groupby(['gender', 'prob'])["dec_o"].mean().reset_index()
prob_vs_dec_o["estimated_probability_pct"] = prob_vs_dec_o["prob"] * 10
prob_vs_dec_o["actual_probability_pct"] = (prob_vs_dec_o["dec_o"] * 100).round(1)
prob_vs_dec_o.drop(columns=["prob", "dec_o"], inplace=True)

fig = go.Figure()
for gender in [0, 1]:
    sub = prob_vs_dec_o[prob_vs_dec_o["gender"] == gender]
    fig.add_trace(
        go.Scatter(
            x=sub['estimated_probability_pct'],
            y=sub['actual_probability_pct'],
            mode='lines+markers',
            line=dict(color=GENDER_COLORS_FROM_LABELS[GENDER_LABELS[gender]], width=2),
            marker=dict(size=10),
            name=f"{GENDER_LABELS[gender]}"
        )
    )
fig.add_trace(go.Scatter(
    x=[0, 100],
    y=[0, 100],
    mode='lines',
    line=dict(color='white', dash='dash'),
    name='Confiance juste'
))
fig.update_layout(
    title=wrap_label("Probabilité estimée vs réelle que le partenaire donne un 'oui'", max_chars=50),
    xaxis_title="Probabilité estimée (%)",
    yaxis_title="Probabilité réelle (%)",
    yaxis_scaleanchor="x",
    yaxis_scaleratio=1,
    width=600,
    height=600,
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.15,
        xanchor='center',
        x=0.5
    )
)
fig.show()

### Bilan partiel — Partie 5

- **Tous les candidats se surévaluent** : +11,5 % pour les femmes, +8,5 % pour les hommes (~1 point sur 10).
- Femmes : surévaluation marquée sur fun (+15,9 %), attirance et sincérité (+13,2 %).
- Hommes : surévaluation principalement sur l'intelligence (+11,6 %).
- **Sur-confiance** aussi dans l'anticipation des "oui" reçus, **plus prononcée chez les hommes** (59 "oui" sur 100 attendus quand ils se disent certains à 100 %, contre 76 chez les femmes).
- Conclusion : les candidats **ne prédisent pas précisément leur propre valeur** sur le marché du dating.

---

## Conclusion et recommandations pour Tinder

Cette étude, menée à partir des données collectées lors d'expériences de speed dating entre 2002 et 2004, avait pour objectif de comprendre ce qui pousse deux personnes à vouloir se revoir. L'analyse révèle un décalage profond entre ce que les candidats déclarent rechercher et ce qui motive réellement leurs décisions.

Sur le plan déclaratif, les femmes mettent en avant l'intelligence et la sincérité, tandis que les hommes valorisent prioritairement l'attirance physique. Mais lorsque l'on observe les décisions concrètes prises à l'issue de chaque speed date, ces préférences s'effacent largement au profit d'un trio commun aux deux genres : l'**attirance physique**, le **fun** et les **intérêts partagés**. À l'inverse, l'intelligence, la sincérité et l'ambition ne jouent qu'un rôle marginal dans la décision réelle. Cet écart entre discours et comportement est central et explique en grande partie pourquoi un profil "bien construit sur le papier" ne suffit pas à générer des matches.

L'étude met également en évidence plusieurs leviers contextuels significatifs : la corrélation des centres d'intérêt augmente sensiblement le taux de match (jusqu'à 20 %), une faible différence d'âge favorise la réciprocité, et un choix restreint de partenaires augmente notablement la propension à dire "oui". Enfin, les candidats — particulièrement les hommes — surestiment systématiquement leurs propres attributs et leur attractivité perçue.

À partir de ces constats, nous formulons les recommandations suivantes à destination de Tinder :

- **Prioriser la qualité des photos et des éléments visuels** dans le profil, l'attirance physique restant le premier déclencheur de l'intérêt.
- **Faire émerger la personnalité "fun"** via des formats *shorts* (vidéos ou photos en situation) plutôt que des biographies textuelles longues.
- **Renforcer l'algorithme de matching sur les centres d'intérêt partagés**, levier plus puissant que les critères socio-démographiques classiques comme l'origine.
- **Limiter volontairement le nombre de profils proposés par session** pour contrer l'effet de paradoxe du choix et stimuler l'engagement.
- **Intégrer un mécanisme de calibration de l'auto-perception**, par exemple via des retours anonymisés, pour aider les utilisateurs à ajuster leur stratégie et réduire la frustration liée à la sur-confiance.